# LeetCode 321 — Create Maximum Number (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxNumber(self, nums1: List[int], nums2: List[int], k: int) -> List[int]:

```

Đề bài nghe rất "đời thường": có hai dãy chữ số, hãy nhặt ra `k` chữ số để ghép thành **số lớn nhất**, với điều kiện các chữ số lấy từ cùng một dãy phải **giữ nguyên thứ tự**. Nhưng đây là một bài **Hard** thật sự, vì nó là **tổ hợp của ba ý tưởng nhỏ**, mỗi ý đều có bẫy riêng:

1. Chọn bao nhiêu chữ số từ dãy 1, bao nhiêu từ dãy 2?
2. Đã biết cần `t` chữ số từ một dãy — lấy những chữ số nào? (**monotonic stack** — ngăn xếp đơn điệu)
3. Có hai dãy con rồi — trộn chúng thế nào cho lớn nhất? (so sánh **phần còn lại**, không chỉ chữ số đầu)

Notebook này đi **từng bậc thang**, từ cách người mới tự nghĩ ra được (thử hết mọi khả năng) tới lời giải chuẩn. Ở mỗi bậc ta trả lời hai câu: *cách cũ vướng ở đâu?* và *mấu chốt nào giúp vượt qua?*

---

## Sau notebook này, bạn sẽ nắm được

1. **Dãy con giữ thứ tự** (subsequence) là gì, cách liệt kê bằng `itertools.combinations`, cách kiểm tra bằng hai con trỏ.
2. **So sánh list theo thứ tự từ điển** (lexicographic order) trong Python: vì sao `[9, 8] > [9, 7, 9]` là `True`, vì sao `[6] < [6, 7]` — và vì sao quy tắc này *khớp hoàn hảo* với bài.
3. **Cách 1 — Brute force**: chọn mọi tập chỉ số, trộn theo mọi cách xen kẽ, lấy max. Rất chậm nhưng **chắc chắn đúng** → dùng làm "trọng tài" để kiểm tra các cách sau.
4. **Cách 2 — Chia bài**: chọn `i` chữ số từ `nums1` và `k - i` từ `nums2`, rồi giải hai bài con:
   - **Bài con 1**: dãy con lớn nhất độ dài `t` của một mảng — từ cách "chọn từng chữ số trong cửa sổ" `O(t·m)` lên **monotonic stack** `O(m)` (họ hàng của LeetCode 402 *Remove K Digits*).
   - **Bài con 2**: trộn hai dãy thành số lớn nhất — vì sao so chữ số đầu là **sai** (phản ví dụ `[6, 7]` và `[6, 0, 4]`), và vì sao so **phần còn lại** `a[i:] > b[j:]` là đúng.
5. **Cách 3 — Ghép lại (cách chính)**: duyệt mọi `i` hợp lệ trong `max(0, k - n) .. min(k, m)`, trace đầy đủ ví dụ 1, phân tích độ phức tạp.
6. **Cách 4 — Tối ưu phép so sánh khi trộn**: so sánh bằng chỉ số, không cắt list.
7. Bản `class Solution` để nộp, test đề + edge case, **đối chiếu chéo với brute force** trên input ngẫu nhiên có `random.seed`.
8. Lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan, 6 bài tập có gợi ý và đáp án ở cuối.

## 1. Đề bài (theo LeetCode)

Bạn được cho hai mảng số nguyên `nums1` và `nums2` có độ dài lần lượt là `m` và `n`. `nums1` và `nums2` biểu diễn **các chữ số** của hai số. Bạn cũng được cho một số nguyên `k`.

Hãy tạo ra **số lớn nhất có độ dài `k <= m + n`** từ các chữ số của hai số đó. **Thứ tự tương đối** của các chữ số lấy từ cùng một mảng phải được giữ nguyên.

Trả về một mảng gồm `k` chữ số biểu diễn đáp án.

**Example 1**

```text
Input:  nums1 = [3,4,6,5], nums2 = [9,1,2,5,8,3], k = 5
Output: [9,8,6,5,3]
```

**Example 2**

```text
Input:  nums1 = [6,7], nums2 = [6,0,4], k = 5
Output: [6,7,6,0,4]
```

**Example 3**

```text
Input:  nums1 = [3,9], nums2 = [8,9], k = 3
Output: [9,8,9]
```

**Constraints**

- `m == nums1.length`
- `n == nums2.length`
- `1 <= m, n <= 500`
- `0 <= nums1[i], nums2[i] <= 9`
- `1 <= k <= m + n`

> Ghi chú: một số phiên bản đề trên LeetCode ghi thêm "`nums1` và `nums2` không có chữ số 0 ở đầu". Điều đó **không ảnh hưởng** gì tới thuật toán dưới đây (thuật toán vẫn đúng kể cả khi có số 0 ở đầu), nên ta không dựa vào nó.

Vài điểm cần đọc kỹ:

- "Số" ở đây được biểu diễn bằng **list chữ số**, ví dụ `[9, 8, 6, 5, 3]` nghĩa là số `98653`. Ta trả về list, không trả về `int`.
- Đáp án luôn có **đúng `k` chữ số**. Vì mọi ứng viên đều dài `k`, "số lớn hơn" chính là "chữ số đầu tiên khác nhau lớn hơn".
- Được lấy từ **cả hai** mảng, xen kẽ tùy ý, nhưng trong **mỗi** mảng phải lấy theo thứ tự trái → phải.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

EXAMPLES = [
    ([3, 4, 6, 5], [9, 1, 2, 5, 8, 3], 5, [9, 8, 6, 5, 3]),
    ([6, 7], [6, 0, 4], 5, [6, 7, 6, 0, 4]),
    ([3, 9], [8, 9], 3, [9, 8, 9]),
]

for nums1, nums2, k, expected in EXAMPLES:
    print(f"nums1={nums1}, nums2={nums2}, k={k}  ->  mong đợi {expected}")

nums1=[3, 4, 6, 5], nums2=[9, 1, 2, 5, 8, 3], k=5  ->  mong đợi [9, 8, 6, 5, 3]
nums1=[6, 7], nums2=[6, 0, 4], k=5  ->  mong đợi [6, 7, 6, 0, 4]
nums1=[3, 9], nums2=[8, 9], k=3  ->  mong đợi [9, 8, 9]


### Hiểu đề bằng tay — Example 1

`nums1 = [3, 4, 6, 5]`, `nums2 = [9, 1, 2, 5, 8, 3]`, `k = 5`, đáp án `[9, 8, 6, 5, 3]`.

Đáp án lấy chữ số từ đâu?

| Chữ số trong đáp án | Lấy từ | Chỉ số |
|---:|---|---:|
| 9 | `nums2` | 0 |
| 8 | `nums2` | 4 |
| 6 | `nums1` | 2 |
| 5 | `nums1` | 3 |
| 3 | `nums2` | 5 |

- Các chỉ số lấy từ `nums1` là `2, 3` — **tăng dần** ✔.
- Các chỉ số lấy từ `nums2` là `0, 4, 5` — **tăng dần** ✔.
- Giữa hai mảng thì xen kẽ thoải mái: lấy 2 chữ số của `nums2`, rồi 2 của `nums1`, rồi 1 của `nums2`.

Vì sao không được `[9, 8, 6, 5, 5]` (lớn hơn đáp án)? Chữ số 5 cuối cùng phải lấy từ đâu đó *sau* những gì đã dùng: `nums1` đã dùng tới chỉ số 3 (hết mảng), `nums2` đã dùng tới chỉ số 4 (chữ số 8), sau đó chỉ còn `3`. Chữ số `5` của `nums2` (chỉ số 3) nằm **trước** chữ số `8` (chỉ số 4) nên không thể dùng sau 8. → Ràng buộc "giữ thứ tự" là thứ khiến bài khó.

Còn so với một cách chọn hợp lệ khác như `[9, 8, 4, 6, 5]`? Hai số khớp nhau `9, 8`, tới chữ số thứ ba `6 > 4` → đáp án thắng.

In [2]:
def show_pick(nums, picked):
    # In mảng, đóng ngoặc [ ] quanh các chữ số được chọn
    return " ".join(f"[{x}]" if i in picked else f" {x} " for i, x in enumerate(nums))

nums1, nums2 = [3, 4, 6, 5], [9, 1, 2, 5, 8, 3]
print("nums1:", show_pick(nums1, {2, 3}))
print("nums2:", show_pick(nums2, {0, 4, 5}))

# Thứ tự lấy chữ số: (tên mảng, chỉ số)
order = [("nums2", 0), ("nums2", 4), ("nums1", 2), ("nums1", 3), ("nums2", 5)]
source = {"nums1": nums1, "nums2": nums2}
print("Ghép lại:", [source[name][i] for name, i in order])

nums1:  3   4  [6] [5]
nums2: [9]  1   2   5  [8] [3]
Ghép lại: [9, 8, 6, 5, 3]


## 2. Kiến thức nền 1 — Dãy con giữ thứ tự (subsequence)

Ba khái niệm hay bị nhầm, xét mảng `nums = [3, 4, 6, 5]`:

| Khái niệm | Nghĩa | Ví dụ hợp lệ | Ví dụ **không** hợp lệ |
|---|---|---|---|
| **subarray** (mảng con liên tiếp) | một đoạn liền nhau | `[4, 6]`, `[3, 4, 6]` | `[3, 6]` (bỏ mất 4 ở giữa) |
| **subsequence** (dãy con) | xóa một số phần tử, **giữ nguyên thứ tự** những phần tử còn lại | `[3, 6]`, `[4, 5]`, `[3, 4, 6, 5]`, `[]` | `[6, 4]` (đảo thứ tự) |
| **subset** (tập con) | chọn phần tử, **không quan tâm** thứ tự | `{6, 4}` | — |

Bài này nói "giữ thứ tự tương đối trong mỗi mảng" → phần lấy từ `nums1` là một **subsequence** của `nums1`, phần lấy từ `nums2` là một **subsequence** của `nums2`.

Cách kiểm tra `sub` có phải dãy con của `arr` không: dùng **hai con trỏ** (two pointers). Con trỏ `p` chạy trên `sub`; ta duyệt `arr` từ trái sang phải, gặp phần tử bằng `sub[p]` thì tăng `p`. Nếu `p` chạy hết `sub` thì đúng là dãy con.

In [3]:
def is_subsequence(sub, arr):
    p = 0                         # con trỏ trên sub: đã khớp được p phần tử đầu
    for x in arr:
        if p < len(sub) and sub[p] == x:
            p += 1
    return p == len(sub)

nums = [3, 4, 6, 5]
for sub in ([3, 6], [4, 5], [6, 4], [], [3, 4, 6, 5], [5, 5]):
    print(f"{str(sub):<14} là dãy con của {nums}? {is_subsequence(sub, nums)}")

[3, 6]         là dãy con của [3, 4, 6, 5]? True
[4, 5]         là dãy con của [3, 4, 6, 5]? True
[6, 4]         là dãy con của [3, 4, 6, 5]? False
[]             là dãy con của [3, 4, 6, 5]? True
[3, 4, 6, 5]   là dãy con của [3, 4, 6, 5]? True
[5, 5]         là dãy con của [3, 4, 6, 5]? False


### Liệt kê mọi dãy con độ dài `t` bằng `itertools.combinations`

`combinations(iterable, t)` sinh ra **mọi cách chọn `t` phần tử**, mỗi cách là một `tuple`, và các phần tử trong tuple luôn **theo đúng thứ tự xuất hiện** trong iterable.

Mẹo: thay vì chọn trên giá trị, ta chọn trên **chỉ số** `range(len(nums))`. Mỗi tuple chỉ số tăng dần ↔ đúng một dãy con giữ thứ tự. Ví dụ với 4 phần tử, chọn 2: có `C(4, 2) = 6` cách:

`(0,1) (0,2) (0,3) (1,2) (1,3) (2,3)`

Ta sẽ dùng nó cho cách brute force. Số cách chọn là `C(m, t)` ("m chọn t", tính bằng `math.comb(m, t)`), tăng **rất nhanh** khi `m` lớn.

In [4]:
from itertools import combinations
from math import comb

nums = [3, 4, 6, 5]
for idx in combinations(range(len(nums)), 2):
    sub = [nums[p] for p in idx]
    print(idx, "->", sub)

print("Số cách chọn 2 trong 4:", comb(4, 2))
print("Số cách chọn 250 trong 500 có", len(str(comb(500, 250))), "chữ số!")

(0, 1) -> [3, 4]
(0, 2) -> [3, 6]
(0, 3) -> [3, 5]
(1, 2) -> [4, 6]
(1, 3) -> [4, 5]
(2, 3) -> [6, 5]
Số cách chọn 2 trong 4: 6
Số cách chọn 250 trong 500 có 150 chữ số!


## 3. Kiến thức nền 2 — So sánh list theo thứ tự từ điển (lexicographic)

Python so sánh hai list **giống như tra từ điển**:

1. So phần tử đầu tiên với phần tử đầu tiên. Khác nhau → phần tử nào lớn hơn thì list đó lớn hơn. **Xong**, không nhìn tiếp.
2. Bằng nhau → so tiếp phần tử thứ hai, thứ ba, ...
3. Nếu một list **hết trước** (nó là *tiền tố* — prefix — của list kia) thì **list ngắn hơn là nhỏ hơn**.

Ví dụ:

| Phép so sánh | Kết quả | Vì sao |
|---|---|---|
| `[9, 8] > [9, 7, 9]` | `True` | vị trí 0: 9 = 9; vị trí 1: **8 > 7** → quyết định luôn, không quan tâm `[9, 7, 9]` dài hơn |
| `[6, 7] > [6, 0, 4]` | `True` | vị trí 1: **7 > 0** |
| `[6] < [6, 7]` | `True` | `[6]` là tiền tố của `[6, 7]` → ngắn hơn thì nhỏ hơn |
| `[] < [0]` | `True` | list rỗng nhỏ hơn mọi list khác rỗng |
| `[1, 0, 0] > [9]` | `False` | vị trí 0: **1 < 9** — dù `100` lớn hơn `9` về giá trị số! |

Dòng cuối là cái bẫy: **khi hai list khác độ dài, so sánh từ điển KHÁC so sánh giá trị số.** Nhưng khi **cùng độ dài** thì hai cách so sánh **trùng nhau** (giống so hai số cùng số chữ số: nhìn chữ số đầu tiên khác nhau). Trong bài, mọi ứng viên đáp án đều dài đúng `k`, nên ta cứ thoải mái dùng `>` và `max()` trên list.

In [5]:
print([9, 8] > [9, 7, 9])        # True
print([6, 7] > [6, 0, 4])        # True
print([6] < [6, 7])              # True: tiền tố ngắn hơn thì nhỏ hơn
print([] < [0])                  # True
print([1, 0, 0] > [9])           # False (dù 100 > 9)

def to_int(digits):
    return int("".join(map(str, digits))) if digits else 0

# Cùng độ dài: so list == so số
a, b = [9, 8, 6, 5, 3], [9, 8, 4, 6, 5]
print(a > b, to_int(a) > to_int(b))

# max() trên list các list cũng dùng đúng quy tắc này
print(max([[5, 1], [4, 9, 9], [5, 0, 9]]))

True
True
True
True
False
True True
[5, 1]


## 4. Kiến thức nền 3 — Trộn xen kẽ (interleave) và đệ quy

Có hai dãy `a` và `b`, một **cách trộn xen kẽ** (interleaving) là một dãy chứa **tất cả** phần tử của cả hai, sao cho thứ tự bên trong `a` và bên trong `b` được giữ nguyên. Ví dụ `a = [1, 2]`, `b = [3]`:

```text
[1, 2, 3]   [1, 3, 2]   [3, 1, 2]
```

(`[2, 1, 3]` thì **không** được vì đảo thứ tự trong `a`.)

Có bao nhiêu cách? Kết quả có `len(a) + len(b)` vị trí, ta chỉ cần chọn vị trí nào dành cho `a` → `C(len(a) + len(b), len(a))` cách. Với `a, b` dài 2 và 1: `C(3, 1) = 3`.

**Liệt kê bằng đệ quy** (recursion — hàm tự gọi lại chính nó với bài toán nhỏ hơn):

- **Trường hợp cơ sở** (base case): nếu `a` rỗng thì chỉ có một cách: chính là `b`; tương tự nếu `b` rỗng.
- **Bước đệ quy**: phần tử đầu tiên của kết quả hoặc là `a[0]`, hoặc là `b[0]`.
  - Chọn `a[0]`: ghép `a[0]` vào trước mọi cách trộn của `a[1:]` với `b`.
  - Chọn `b[0]`: ghép `b[0]` vào trước mọi cách trộn của `a` với `b[1:]`.

Mỗi lần gọi, tổng độ dài giảm 1, nên đệ quy chắc chắn dừng.

In [6]:
def all_interleavings(a, b):
    if not a:
        return [list(b)]
    if not b:
        return [list(a)]
    result = []
    for rest in all_interleavings(a[1:], b):      # chữ số đầu lấy từ a
        result.append([a[0]] + rest)
    for rest in all_interleavings(a, b[1:]):      # chữ số đầu lấy từ b
        result.append([b[0]] + rest)
    return result

print(all_interleavings([1, 2], [3]))
print("Số cách trộn [6,7] với [6,0,4]:", len(all_interleavings([6, 7], [6, 0, 4])), "= C(5,2) =", comb(5, 2))
print("Lớn nhất:", max(all_interleavings([6, 7], [6, 0, 4])))

[[1, 2, 3], [1, 3, 2], [3, 1, 2]]
Số cách trộn [6,7] với [6,0,4]: 10 = C(5,2) = 10
Lớn nhất: [6, 7, 6, 0, 4]


## 5. Cách 1 — Brute force: thử TẤT CẢ (làm trọng tài)

Cách người mới nghĩ ra đầu tiên: *"không biết chọn thế nào thì thử hết"*.

1. Với mỗi `i` = số chữ số lấy từ `nums1` (từ `0` tới `k`), phần còn lại `k - i` lấy từ `nums2`.
2. Chọn mọi tập chỉ số `idx1` gồm `i` chỉ số của `nums1` (`combinations`) → dãy con `a`.
3. Chọn mọi tập chỉ số `idx2` gồm `k - i` chỉ số của `nums2` → dãy con `b`.
4. Trộn `a` và `b` theo **mọi** cách xen kẽ → mỗi cách là một ứng viên dài `k`.
5. Giữ ứng viên lớn nhất (so list theo thứ tự từ điển).

Một chi tiết tiện lợi: nếu `i > m` thì `combinations(range(m), i)` **không sinh ra gì cả** (không thể chọn 5 phần tử trong 4), nên ta không cần tự lọc `i` không hợp lệ — vòng lặp tự bỏ qua.

Cách này **chắc chắn đúng** vì nó xét mọi khả năng hợp lệ. Ta không nộp nó, nhưng sẽ dùng nó làm **trọng tài** (oracle) để kiểm tra các cách nhanh.

In [7]:
def max_number_bruteforce(nums1: List[int], nums2: List[int], k: int) -> List[int]:
    m, n = len(nums1), len(nums2)
    best = []                                   # [] nhỏ hơn mọi list khác rỗng
    for i in range(k + 1):                      # i chữ số từ nums1, k - i từ nums2
        for idx1 in combinations(range(m), i):
            a = [nums1[p] for p in idx1]
            for idx2 in combinations(range(n), k - i):
                b = [nums2[p] for p in idx2]
                for cand in all_interleavings(a, b):
                    if cand > best:
                        best = cand
    return best

for nums1, nums2, k, expected in EXAMPLES:
    got = max_number_bruteforce(nums1, nums2, k)
    print(got, "OK" if got == expected else f"SAI, mong đợi {expected}")
    assert got == expected

[9, 8, 6, 5, 3] OK
[6, 7, 6, 0, 4] OK
[9, 8, 9] OK


### Độ phức tạp cách 1 và chỗ vướng

Với mỗi `i`, số ứng viên brute force tạo ra là:

```text
C(m, i)  ×  C(n, k - i)  ×  C(k, i)
 chọn a      chọn b        cách trộn
```

Cộng lại trên mọi `i`. Với Example 1 (`m = 4, n = 6, k = 5`) chỉ khoảng hai nghìn ứng viên — chạy tức thì. Nhưng các hệ số tổ hợp **bùng nổ** cực nhanh: với `m = n = 500` thì con số có hàng trăm chữ số, vũ trụ kết thúc cũng chưa chạy xong.

Chỗ vướng nằm ở chỗ: brute force coi **mọi** dãy con và **mọi** cách trộn là "có thể là đáp án", trong khi phần lớn rõ ràng là tệ (ví dụ dãy con bắt đầu bằng `1` khi có thể bắt đầu bằng `9`). Ta cần những quan sát giúp **loại bỏ hàng loạt** ứng viên mà không cần xét.

In [8]:
def count_candidates(m, n, k):
    total = 0
    for i in range(max(0, k - n), min(k, m) + 1):
        total += comb(m, i) * comb(n, k - i) * comb(k, i)
    return total

print("Example 1 (4, 6, 5):", count_candidates(4, 6, 5), "ứng viên")
print("(10, 10, 10):      ", count_candidates(10, 10, 10), "ứng viên")
big = count_candidates(500, 500, 500)
print("(500, 500, 500):    một số có", len(str(big)), "chữ số")

Example 1 (4, 6, 5): 2136 ứng viên
(10, 10, 10):       38165260 ứng viên
(500, 500, 500):    một số có 449 chữ số


## 6. Bước đột phá — Chia bài toán theo `i`

Nhìn lại đáp án của Example 1: nó gồm **2** chữ số từ `nums1` (`[6, 5]`) và **3** chữ số từ `nums2` (`[9, 8, 3]`), trộn lại. Tổng quát:

> Đáp án = trộn( một dãy con độ dài `i` của `nums1`, một dãy con độ dài `k - i` của `nums2` ) với một `i` nào đó.

Ta không biết trước `i` → **thử mọi `i`**. Nhưng `i` không được tùy ý:

- `0 <= i <= m` (không lấy quá số chữ số `nums1` có) và `i <= k`.
- `0 <= k - i <= n` ⟺ `i >= k - n` (phần lấy từ `nums2` không được vượt quá `n`).

Gộp lại: `max(0, k - n) <= i <= min(k, m)`.

Ví dụ:

| Ví dụ | `m` | `n` | `k` | `max(0, k-n)` | `min(k, m)` | Các `i` hợp lệ |
|---|---:|---:|---:|---:|---:|---|
| 1 | 4 | 6 | 5 | 0 | 4 | `0, 1, 2, 3, 4` |
| 2 | 2 | 3 | 5 | 2 | 2 | chỉ `2` (phải lấy hết) |
| 3 | 2 | 2 | 3 | 1 | 2 | `1, 2` |

Số giá trị `i` tối đa là khoảng `min(m, n) + 1 <= 501` — ít, thử hết được.

In [9]:
def valid_i_range(m, n, k):
    return range(max(0, k - n), min(k, m) + 1)

for nums1, nums2, k, _ in EXAMPLES:
    r = valid_i_range(len(nums1), len(nums2), k)
    print(f"m={len(nums1)}, n={len(nums2)}, k={k}: i chạy {list(r)}")

m=4, n=6, k=5: i chạy [0, 1, 2, 3, 4]
m=2, n=3, k=5: i chạy [2]
m=2, n=2, k=3: i chạy [1, 2]


### Quan sát then chốt: với `i` cố định, cứ lấy **dãy con lớn nhất** của mỗi bên

Đã cố định `i`. Có rất nhiều dãy con độ dài `i` của `nums1`. Câu hỏi: lấy dãy nào? Trực giác: lấy dãy **lớn nhất** (theo thứ tự từ điển). Vì sao điều này an toàn?

**Lập luận "cùng một kiểu trộn".** Một cách trộn có thể mô tả bằng một chuỗi chữ `A`/`B` dài `k`: ký tự thứ `p` nói chữ số thứ `p` của kết quả lấy tiếp từ `a` hay từ `b`. Ví dụ `a = [6, 5]`, `b = [9, 8, 3]`, kiểu trộn `"BBAAB"` cho `[9, 8, 6, 5, 3]`.

Giả sử có hai dãy con cùng độ dài `a > a'` (ví dụ `a = [6, 5]`, `a' = [6, 4]`). Lấy **cùng một kiểu trộn** cho `a` và `a'` (với cùng `b`):

- Các chữ số lấy từ `b` giống hệt nhau.
- Các chữ số lấy từ `a` / `a'` giống nhau cho tới vị trí đầu tiên mà `a` và `a'` khác nhau — ở đó `a` lớn hơn.
- Vậy hai kết quả trùng nhau cho tới đúng chỗ chữ số đó được đặt vào, và tại đó kết quả của `a` **lớn hơn**.

Tức là: *với bất kỳ cách trộn nào của `a'`, cách trộn cùng kiểu của `a` đều lớn hơn hoặc bằng* → **cách trộn tốt nhất của `a` ≥ cách trộn tốt nhất của `a'`**. Lập luận y hệt cho `b`. Kết luận:

> Với mỗi `i`, chỉ cần xét **một** cặp: `a = dãy con lớn nhất độ dài i của nums1`, `b = dãy con lớn nhất độ dài k − i của nums2`.

Từ hàng tỉ cặp `(a, b)` còn đúng **một** cặp cho mỗi `i`. Cell dưới kiểm chứng bằng số.

In [10]:
def apply_pattern(a, b, pattern):
    # pattern là chuỗi 'A'/'B': mỗi ký tự nói chữ số tiếp theo lấy từ a hay từ b
    ia = ib = 0
    out = []
    for c in pattern:
        if c == "A":
            out.append(a[ia]); ia += 1
        else:
            out.append(b[ib]); ib += 1
    return out

b = [9, 8, 3]
for a in ([6, 5], [6, 4], [4, 6]):          # [6,5] > [6,4] > [4,6]
    print(f"a={a}: kiểu 'BBAAB' -> {apply_pattern(a, b, 'BBAAB')}, "
          f"trộn tốt nhất -> {max(all_interleavings(a, b))}")

a=[6, 5]: kiểu 'BBAAB' -> [9, 8, 6, 5, 3], trộn tốt nhất -> [9, 8, 6, 5, 3]
a=[6, 4]: kiểu 'BBAAB' -> [9, 8, 6, 4, 3], trộn tốt nhất -> [9, 8, 6, 4, 3]
a=[4, 6]: kiểu 'BBAAB' -> [9, 8, 4, 6, 3], trộn tốt nhất -> [9, 8, 4, 6, 3]


### Lộ trình: bài lớn = vòng lặp `i` + hai bài con

```text
best = []
với mỗi i trong max(0, k-n) .. min(k, m):
    a = max_subsequence(nums1, i)         <- Bài con 1
    b = max_subsequence(nums2, k - i)     <- Bài con 1
    cand = merge_max(a, b)                <- Bài con 2
    best = max(best, cand)
```

- **Bài con 1** (mục 7): cho một mảng và `t`, tìm dãy con độ dài `t` lớn nhất.
- **Bài con 2** (mục 8): cho hai dãy, trộn **toàn bộ** chúng thành dãy lớn nhất.

Mỗi bài con ta lại đi từ cách dễ nghĩ → cách tốt.

## 7. Bài con 1 — Dãy con lớn nhất độ dài `t` của một mảng

**Đề bài con:** cho `nums` (dài `m`) và `0 <= t <= m`, tìm dãy con giữ thứ tự, độ dài `t`, lớn nhất theo thứ tự từ điển.

Ví dụ: `nums = [9, 1, 2, 5, 8, 3]`, `t = 3` → `[9, 8, 3]`.

### Bước 1a — Chọn từng chữ số trong một "cửa sổ" (greedy)

Muốn số lớn nhất thì **chữ số đầu tiên phải lớn nhất có thể** (vì so sánh từ điển quyết định ngay ở chữ số đầu khác nhau). Nhưng không được tham lam mù quáng: sau chữ số đầu ta còn phải lấy thêm `t - 1` chữ số nữa, nên chữ số đầu **không được nằm quá gần cuối mảng**.

Chữ số thứ `p` (đếm từ 0) phải chừa lại ít nhất `t - p - 1` chữ số phía sau → chỉ số của nó tối đa là `m - (t - p)`. Vậy:

- Cửa sổ cho chữ số thứ `p`: từ `start` (ngay sau chữ số vừa lấy) tới `end = m - (t - p)`.
- Lấy **max** trong cửa sổ; nếu max xuất hiện nhiều lần, lấy lần xuất hiện **bên trái nhất** — vì nó chừa lại nhiều lựa chọn hơn cho các bước sau. Ví dụ `[9, 9, 1]`, `t = 2`: lấy số 9 ở chỉ số 0 → bước sau còn `[9, 1]` → được `[9, 9]`; nếu lấy số 9 ở chỉ số 1 thì bước sau chỉ còn `[1]` → `[9, 1]`, tệ hơn.

Trace `nums = [9, 1, 2, 5, 8, 3]`, `t = 3` (`m = 6`):

| Chữ số thứ | Cửa sổ chỉ số | Cửa sổ | Lấy | Kết quả |
|---:|---|---|---|---|
| 1 | `0..3` | `[9, 1, 2, 5]` | **9** (chỉ số 0) | `[9]` |
| 2 | `1..4` | `[1, 2, 5, 8]` | **8** (chỉ số 4) | `[9, 8]` |
| 3 | `5..5` | `[3]` | **3** (chỉ số 5) | `[9, 8, 3]` |

Bước 2: cửa sổ `1..4` bỏ qua `1, 2, 5` để lấy `8` — đúng như mong muốn, vì `[9, 8, ...]` thắng mọi `[9, 5, ...]`.

In [11]:
def max_subseq_bruteforce(nums, t):
    # Trọng tài: thử mọi dãy con độ dài t
    return max([nums[p] for p in idx] for idx in combinations(range(len(nums)), t))

def max_subseq_window(nums, t):
    m = len(nums)
    res, start = [], 0
    for p in range(t):
        end = m - (t - p)                 # chừa lại t - p - 1 chữ số phía sau
        window = nums[start:end + 1]
        best = max(window)
        pos = start + window.index(best)  # .index trả về lần xuất hiện bên trái nhất
        res.append(best)
        start = pos + 1
    return res

for nums, t in [([9, 1, 2, 5, 8, 3], 3), ([9, 9, 1], 2), ([3, 4, 6, 5], 2), ([5, 1], 0)]:
    print(nums, t, "->", max_subseq_window(nums, t), "| brute:", max_subseq_bruteforce(nums, t))

[9, 1, 2, 5, 8, 3] 3 -> [9, 8, 3] | brute: [9, 8, 3]
[9, 9, 1] 2 -> [9, 9] | brute: [9, 9]
[3, 4, 6, 5] 2 -> [6, 5] | brute: [6, 5]
[5, 1] 0 -> [] | brute: []


In [12]:
def trace_window(nums, t):
    m = len(nums)
    res, start = [], 0
    print(f"nums = {nums}, t = {t}, m = {m}")
    for p in range(t):
        end = m - (t - p)
        window = nums[start:end + 1]
        best = max(window)
        pos = start + window.index(best)
        res.append(best)
        print(f"  chữ số thứ {p + 1}: cửa sổ {start}..{end} = {window} -> lấy {best} (chỉ số {pos}), res = {res}")
        start = pos + 1
    return res

trace_window([9, 1, 2, 5, 8, 3], 3)

nums = [9, 1, 2, 5, 8, 3], t = 3, m = 6
  chữ số thứ 1: cửa sổ 0..3 = [9, 1, 2, 5] -> lấy 9 (chỉ số 0), res = [9]
  chữ số thứ 2: cửa sổ 1..4 = [1, 2, 5, 8] -> lấy 8 (chỉ số 4), res = [9, 8]
  chữ số thứ 3: cửa sổ 5..5 = [3] -> lấy 3 (chỉ số 5), res = [9, 8, 3]


[9, 8, 3]

### Bước 1a vướng ở đâu? Và đổi góc nhìn: "chọn `t`" = "xóa `m − t`"

Mỗi chữ số ta quét một cửa sổ dài tới `m - t + 1` phần tử → tổng `O(t · (m − t + 1))`, xấu nhất cỡ `O(m²)` (khi `t ≈ m/2`). Bài lớn gọi bài con này cho mỗi `i` (tới ~500 lần) trên cả hai mảng → có thể tới hàng trăm triệu phép so sánh trong Python. Chậm.

Vấn đề: các cửa sổ liên tiếp **chồng lên nhau rất nhiều**, và ta quét lại chúng từ đầu mỗi lần. Ta muốn **quét mảng đúng một lần**.

**Đổi góc nhìn:** chọn giữ lại `t` chữ số cũng chính là **xóa đi `drop = m − t` chữ số**. Khi nào nên xóa một chữ số? Xét `[9, 1, 2]`, phải xóa 1 chữ số:

- Xóa `9` → `[1, 2]`; xóa `1` → `[9, 2]`; xóa `2` → `[9, 1]`. Tốt nhất là xóa **`1`** — chữ số đầu tiên **nhỏ hơn chữ số đứng ngay sau nó**. Xóa nó thì chữ số lớn hơn phía sau "tiến lên" đúng vị trí đó, làm số lớn lên.

Quy tắc: *"đang thấy chữ số `x`, nếu chữ số giữ lại ngay trước nó nhỏ hơn `x` và ta vẫn còn quyền xóa → xóa chữ số trước đó"*, lặp lại được nhiều lần. Chỉ cần nhìn vào **chữ số giữ lại gần nhất** → đúng công việc của **stack**.

Đây chính là ý tưởng của **LeetCode 402 — Remove K Digits** (xóa `k` chữ số để được số **nhỏ nhất**); bài ta làm ngược lại: muốn **lớn nhất** nên đảo chiều so sánh.

### Kiến thức nền 4 — Stack bằng list Python

**Stack** (ngăn xếp) giống một chồng đĩa: chỉ đặt thêm lên **đỉnh** và lấy ra từ **đỉnh** (LIFO — *Last In, First Out*: vào sau ra trước). Trong Python, list làm stack rất tốt:

| Thao tác | Code | Chi phí |
|---|---|---|
| đẩy `x` lên đỉnh (push) | `stack.append(x)` | `O(1)` |
| lấy phần tử đỉnh ra (pop) | `stack.pop()` | `O(1)` |
| xem đỉnh mà không lấy ra | `stack[-1]` | `O(1)` |
| stack có rỗng không? | `if stack:` / `if not stack:` | `O(1)` |

**Monotonic stack** (ngăn xếp đơn điệu): trước khi đẩy `x`, ta `pop` những phần tử ở đỉnh "vi phạm thứ tự" so với `x`. Nhờ vậy các phần tử trong stack luôn giữ một tính chất (ở bài này: **không tăng** từ đáy lên đỉnh, chừng nào còn quyền xóa).

In [13]:
stack = []
for x in [5, 3, 8]:
    stack.append(x)
    print("push", x, "->", stack)

print("đỉnh hiện tại:", stack[-1])
print("pop ra:", stack.pop(), "->", stack)
print("stack rỗng?", not stack)

push 5 -> [5]
push 3 -> [5, 3]
push 8 -> [5, 3, 8]
đỉnh hiện tại: 8
pop ra: 8 -> [5, 3]
stack rỗng? False


### Bước 1b — Monotonic stack cho dãy con lớn nhất

```python
drop = len(nums) - t          # số chữ số được phép xóa
stack = []
for x in nums:
    while drop > 0 and stack and stack[-1] < x:
        stack.pop()           # chữ số nhỏ đứng trước chữ số lớn hơn -> xóa nó
        drop -= 1
    stack.append(x)
return stack[:t]              # nếu còn thừa quyền xóa, xóa ở cuối
```

Giải thích từng điều kiện của `while`:

- `drop > 0`: còn quyền xóa. Nếu không kiểm tra, ta có thể xóa quá tay và kết quả ngắn hơn `t`.
- `stack`: stack khác rỗng thì mới có `stack[-1]` để xem.
- `stack[-1] < x`: chữ số giữ lại gần nhất **nhỏ hơn hẳn** `x` → xóa nó thì `x` tiến lên, số lớn hơn. Dùng `<` chứ **không** dùng `<=`: nếu bằng nhau, xóa đi rồi đặt lại một chữ số y hệt thì số không lớn thêm mà lại **phí một quyền xóa** — có thể làm hỏng kết quả (xem mục Lỗi hay gặp: `[5, 5, 3]`, `t = 2`).

Vì sao cần `stack[:t]` ở cuối? Nếu mảng **không tăng** ở đoạn cuối (ví dụ `[9, 7, 6]`), không bao giờ có lý do để pop, `drop` còn dư, stack dài hơn `t`. Khi đó chữ số nên xóa là những chữ số **cuối cùng** (xóa ở đuôi làm số mất chữ số nhỏ nhất về vị trí) → cắt `stack[:t]`.

In [14]:
def max_subsequence(nums: List[int], t: int) -> List[int]:
    drop = len(nums) - t
    stack = []
    for x in nums:
        while drop > 0 and stack and stack[-1] < x:
            stack.pop()
            drop -= 1
        stack.append(x)
    return stack[:t]

print(max_subsequence([9, 1, 2, 5, 8, 3], 3))   # [9, 8, 3]
print(max_subsequence([3, 4, 6, 5], 2))         # [6, 5]
print(max_subsequence([8, 9, 7, 6], 2))         # [9, 7]
print(max_subsequence([3, 9], 0))               # []

[9, 8, 3]
[6, 5]
[9, 7]
[]


### Trace monotonic stack

**Trace 1:** `nums = [9, 1, 2, 5, 8, 3]`, `t = 3` → `drop = 3`. Cột "stack sau" là trạng thái **sau khi** đã pop và push `x`.

| i | x | pop | stack sau | drop còn |
|---:|---:|---|---|---:|
| 0 | 9 | — | `[9]` | 3 |
| 1 | 1 | — | `[9, 1]` | 3 |
| 2 | 2 | pop `[1]` | `[9, 2]` | 2 |
| 3 | 5 | pop `[2]` | `[9, 5]` | 1 |
| 4 | 8 | pop `[5]` | `[9, 8]` | 0 |
| 5 | 3 | — | `[9, 8, 3]` | 0 |

Quan sát: tại `x = 8`, stack là `[9, 5]`, đỉnh `5 < 8` → pop `5` (hết quyền xóa). `9 < 8` sai nên dừng. Kết quả `[9, 8, 3]`, trùng với cách cửa sổ — nhưng chỉ quét mảng **một lần**.

**Trace 2 (có cắt đuôi):** `nums = [8, 9, 7, 6]`, `t = 2` → `drop = 2`.

| i | x | pop | stack sau | drop còn |
|---:|---:|---|---|---:|
| 0 | 8 | — | `[8]` | 2 |
| 1 | 9 | pop `[8]` | `[9]` | 1 |
| 2 | 7 | — | `[9, 7]` | 1 |
| 3 | 6 | — | `[9, 7, 6]` | 1 |

Kết thúc stack là `[9, 7, 6]` (dài 3 > `t = 2`) vì `7, 6` giảm dần, không có lý do pop. Còn dư 1 quyền xóa → cắt đuôi: `stack[:2] = [9, 7]`.

In [15]:
def trace_max_subsequence(nums, t):
    drop = len(nums) - t
    stack = []
    print(f"nums = {nums}, t = {t}, drop ban đầu = {drop}")
    for i, x in enumerate(nums):
        popped = []
        while drop > 0 and stack and stack[-1] < x:
            popped.append(stack.pop())
            drop -= 1
        stack.append(x)
        note = f"pop {popped}" if popped else "không pop"
        print(f"  i={i}, x={x}: {note:<12} -> stack = {stack}, drop còn {drop}")
    print(f"  stack[:{t}] = {stack[:t]}")
    return stack[:t]

trace_max_subsequence([9, 1, 2, 5, 8, 3], 3)
print()
trace_max_subsequence([8, 9, 7, 6], 2)

nums = [9, 1, 2, 5, 8, 3], t = 3, drop ban đầu = 3
  i=0, x=9: không pop    -> stack = [9], drop còn 3
  i=1, x=1: không pop    -> stack = [9, 1], drop còn 3
  i=2, x=2: pop [1]      -> stack = [9, 2], drop còn 2
  i=3, x=5: pop [2]      -> stack = [9, 5], drop còn 1
  i=4, x=8: pop [5]      -> stack = [9, 8], drop còn 0
  i=5, x=3: không pop    -> stack = [9, 8, 3], drop còn 0
  stack[:3] = [9, 8, 3]

nums = [8, 9, 7, 6], t = 2, drop ban đầu = 2
  i=0, x=8: không pop    -> stack = [8], drop còn 2
  i=1, x=9: pop [8]      -> stack = [9], drop còn 1
  i=2, x=7: không pop    -> stack = [9, 7], drop còn 1
  i=3, x=6: không pop    -> stack = [9, 7, 6], drop còn 1
  stack[:2] = [9, 7]


[9, 7]

### Vì sao monotonic stack đúng, và vì sao `O(m)`?

**Vì sao đúng (bằng lời).** Bài "xóa `drop` chữ số để số lớn nhất" có một quy tắc tham lam đã thấy ở trên: *xóa chữ số đầu tiên nhỏ hơn chữ số đứng ngay sau nó; nếu không có chữ số như vậy (dãy không tăng) thì xóa chữ số cuối*. Mỗi lần xóa như vậy cho số lớn nhất có thể với **một** lần xóa, và làm lặp lại `drop` lần sẽ ra tối ưu (cùng lập luận với LeetCode 402, chỉ đảo chiều so sánh).

Stack thực hiện đúng quy tắc đó nhưng **không quét lại từ đầu**:

- **Invariant**: tại mọi thời điểm, stack (từ đáy lên đỉnh) là phần đã giữ lại của đoạn đã đọc; nó **không tăng** ở mọi chỗ trừ khi đã hết quyền xóa. Vì vậy "chữ số đầu tiên nhỏ hơn chữ số sau nó" chỉ có thể là **đỉnh stack** so với `x` mới tới — không cần nhìn sâu hơn.
- Khi hết mảng mà còn quyền xóa, stack không tăng → xóa ở cuối → `stack[:t]`.

**Vì sao `O(m)` dù có `while` lồng trong `for`?** Mỗi phần tử được `append` **đúng một lần** và `pop` **nhiều nhất một lần**. Tổng số lần `pop` của **cả** thuật toán ≤ `m` (thậm chí ≤ `drop`). Vậy tổng công việc ≤ `2m` → `O(m)` thời gian, `O(m)` bộ nhớ cho stack. Đây là phân tích **amortized** (khấu hao): một vòng `while` có thể dài, nhưng tổng của mọi vòng `while` bị chặn.

Cell dưới đối chiếu cả 3 cách của bài con 1 trên nhiều mảng ngẫu nhiên (seed cố định) — dùng chữ số `0..3` để có **nhiều chữ số trùng**, là chỗ dễ sai nhất.

In [16]:
import random

random.seed(321)
for trial in range(2000):
    nums = [random.randint(0, 3) for _ in range(random.randint(1, 8))]
    t = random.randint(0, len(nums))
    ref = max_subseq_bruteforce(nums, t)
    assert max_subseq_window(nums, t) == ref, (nums, t)
    assert max_subsequence(nums, t) == ref, (nums, t)
print("Bài con 1: brute force, cửa sổ, monotonic stack khớp nhau trên 2000 ca ngẫu nhiên.")

Bài con 1: brute force, cửa sổ, monotonic stack khớp nhau trên 2000 ca ngẫu nhiên.


## 8. Bài con 2 — Trộn hai dãy thành số lớn nhất

**Đề bài con:** cho hai dãy `a`, `b`, tạo dãy dài `len(a) + len(b)` chứa **tất cả** chữ số của cả hai, giữ thứ tự trong mỗi dãy, sao cho lớn nhất.

### Bước 2a — Cách tự nhiên: so chữ số đầu (giống merge trong merge sort)

Hai con trỏ `i` trên `a`, `j` trên `b`. Mỗi bước nhìn `a[i]` và `b[j]`, lấy cái lớn hơn. Khi một dãy hết thì lấy nốt dãy kia. Nghe rất hợp lý — chữ số lớn đặt trước thì số lớn.

**Nhưng khi `a[i] == b[j]` thì lấy bên nào?** Thử `a = [6, 7]`, `b = [6, 0, 4]`:

- Lấy `6` của `b` trước: còn `a = [6, 7]`, `b = [0, 4]` → `6, 7, 0, 4` → kết quả `[6, 6, 7, 0, 4]`.
- Lấy `6` của `a` trước: còn `a = [7]`, `b = [6, 0, 4]` → `7, 6, 0, 4` → kết quả `[6, 7, 6, 0, 4]` — **lớn hơn**.

Nếu quy ước "bằng nhau thì lấy `a`", ví dụ này đúng, nhưng chỉ cần **đổi chỗ** hai dãy (`a = [6, 0, 4]`, `b = [6, 7]`) là sai. **Không có quy ước cố định nào cho trường hợp bằng nhau là đúng** — phải nhìn xa hơn chữ số đầu.

In [17]:
def merge_head_only(a, b):
    # SAI: chỉ so chữ số đầu, bằng nhau thì ưu tiên a
    i = j = 0
    res = []
    while i < len(a) or j < len(b):
        if j == len(b) or (i < len(a) and a[i] >= b[j]):
            res.append(a[i]); i += 1
        else:
            res.append(b[j]); j += 1
    return res

for a, b in [([6, 7], [6, 0, 4]), ([6, 0, 4], [6, 7])]:
    print(f"a={a}, b={b}: so đầu -> {merge_head_only(a, b)}, "
          f"đúng (brute) -> {max(all_interleavings(a, b))}")

a=[6, 7], b=[6, 0, 4]: so đầu -> [6, 7, 6, 0, 4], đúng (brute) -> [6, 7, 6, 0, 4]
a=[6, 0, 4], b=[6, 7]: so đầu -> [6, 6, 7, 0, 4], đúng (brute) -> [6, 7, 6, 0, 4]


### Bước 2b — Mấu chốt: so sánh **PHẦN CÒN LẠI** `a[i:] > b[j:]`

Thay vì so `a[i]` với `b[j]`, ta so **toàn bộ phần chưa dùng** của hai dãy theo thứ tự từ điển:

```python
if a[i:] > b[j:]:
    lấy a[i]
else:
    lấy b[j]
```

Vì sao hợp lý? Có hai tình huống:

1. **Chữ số đầu khác nhau** (`a[i] != b[j]`): so từ điển quyết định ngay ở chữ số đầu → giống hệt cách 2a, lấy chữ số lớn hơn. Không có gì thay đổi.
2. **Chữ số đầu bằng nhau**: lấy bên nào thì chữ số ghi ra lúc này cũng như nhau. Khác biệt là **dãy nào được "tiến lên"** để lộ ra chữ số tiếp theo. So từ điển tự động nhìn tiếp tới **chỗ khác nhau đầu tiên**: dãy nào ở đó có chữ số lớn hơn thì ta nên "tiêu" dãy đó trước, để chữ số lớn kia **được mở khóa sớm hơn**.
   - `[6, 7] > [6, 0, 4]` (vì `7 > 0`) → lấy `6` của `[6, 7]` → chữ số `7` lộ ra ngay ở bước sau.
   - `[6, 7, 1]` vs `[6, 7, 9]`: bằng nhau tới 2 chữ số, khác ở `1 < 9` → lấy từ `[6, 7, 9]` trước. So chữ số đầu (hay so "hai chữ số đầu") đều không thấy được điều này.

**Trường hợp tiền tố.** `a[i:] = [6]`, `b[j:] = [6, 7]`. Python nói `[6] < [6, 7]` → lấy từ `b`: ra `6`, rồi `[6]` vs `[7]` → `7`, rồi `6` → `[6, 7, 6]`. Nếu lấy từ `a` trước: `6`, rồi `[]` vs `[6, 7]` → `6, 7` → `[6, 6, 7]` — nhỏ hơn. Vậy quy tắc "list ngắn hơn (là tiền tố) thì nhỏ hơn" của Python lại **đúng ý** ta: dãy dài hơn còn "giấu" thêm chữ số phía sau, nên tiêu nó trước.

Khi `a[i:] == b[j:]` (giống hệt nhau) thì lấy bên nào cũng được; code dưới lấy `b`.

In [18]:
print([6, 7] > [6, 0, 4])        # True  -> lấy từ a
print([6, 7, 1] > [6, 7, 9])     # False -> lấy từ b
print([6] > [6, 7])              # False -> lấy từ b (dãy dài hơn)

def merge_max(a: List[int], b: List[int]) -> List[int]:
    i = j = 0
    res = []
    while i < len(a) or j < len(b):
        if a[i:] > b[j:]:        # so sánh phần còn lại; a hết thì a[i:] == [] luôn thua
            res.append(a[i]); i += 1
        else:
            res.append(b[j]); j += 1
    return res

print(merge_max([6, 7], [6, 0, 4]), merge_max([6, 0, 4], [6, 7]))
print(merge_max([6, 5], [9, 8, 3]))

True
False
False
[6, 7, 6, 0, 4] [6, 7, 6, 0, 4]
[9, 8, 6, 5, 3]


Để ý dòng `if a[i:] > b[j:]` cũng tự xử lý luôn chuyện "một dãy đã hết": khi `i == len(a)` thì `a[i:] == []`, mà `[]` nhỏ hơn hoặc bằng mọi list → luôn lấy từ `b` (lúc đó `b` chắc chắn còn, vì vòng `while` còn chạy). Khi `j == len(b)` thì `b[j:] == []`, còn `a[i:]` khác rỗng nên `a[i:] > []` → lấy từ `a`. Không cần `if` riêng cho dãy đã hết.

### Trace merge

**Trace 1:** `a = [6, 7]`, `b = [6, 0, 4]` (Example 2).

| Bước | `a[i:]` | `b[j:]` | So sánh | Lấy | Kết quả |
|---:|---|---|---|---|---|
| 1 | `[6, 7]` | `[6, 0, 4]` | `a[i:] > b[j:]` | a → **6** | `[6]` |
| 2 | `[7]` | `[6, 0, 4]` | `a[i:] > b[j:]` | a → **7** | `[6, 7]` |
| 3 | `[]` | `[6, 0, 4]` | `a[i:] <= b[j:]` | b → **6** | `[6, 7, 6]` |
| 4 | `[]` | `[0, 4]` | `a[i:] <= b[j:]` | b → **0** | `[6, 7, 6, 0]` |
| 5 | `[]` | `[4]` | `a[i:] <= b[j:]` | b → **4** | `[6, 7, 6, 0, 4]` |

**Trace 2 (bằng nhau tới 2 chữ số):** `a = [6, 7, 1]`, `b = [6, 7, 9]`.

| Bước | `a[i:]` | `b[j:]` | So sánh | Lấy | Kết quả |
|---:|---|---|---|---|---|
| 1 | `[6, 7, 1]` | `[6, 7, 9]` | `a[i:] <= b[j:]` | b → **6** | `[6]` |
| 2 | `[6, 7, 1]` | `[7, 9]` | `a[i:] <= b[j:]` | b → **7** | `[6, 7]` |
| 3 | `[6, 7, 1]` | `[9]` | `a[i:] <= b[j:]` | b → **9** | `[6, 7, 9]` |
| 4 | `[6, 7, 1]` | `[]` | `a[i:] > b[j:]` | a → **6** | `[6, 7, 9, 6]` |
| 5 | `[7, 1]` | `[]` | `a[i:] > b[j:]` | a → **7** | `[6, 7, 9, 6, 7]` |
| 6 | `[1]` | `[]` | `a[i:] > b[j:]` | a → **1** | `[6, 7, 9, 6, 7, 1]` |

Ở Trace 2, ba bước đầu đều lấy từ `b`: nhờ vậy chữ số `9` xuất hiện ở vị trí thứ 3. Nếu lấy `6` của `a` trước thì `9` bị đẩy lùi về sau.

In [19]:
def trace_merge(a, b):
    i = j = 0
    res = []
    print(f"merge a={a}, b={b}")
    while i < len(a) or j < len(b):
        ra, rb = a[i:], b[j:]
        if ra > rb:
            res.append(a[i]); i += 1; who = "a"
        else:
            res.append(b[j]); j += 1; who = "b"
        print(f"  {str(ra):>10} vs {str(rb):<10} -> lấy {res[-1]} từ {who}, res = {res}")
    return res

trace_merge([6, 7], [6, 0, 4])
print()
trace_merge([6, 7, 1], [6, 7, 9])

merge a=[6, 7], b=[6, 0, 4]
      [6, 7] vs [6, 0, 4]  -> lấy 6 từ a, res = [6]
         [7] vs [6, 0, 4]  -> lấy 7 từ a, res = [6, 7]
          [] vs [6, 0, 4]  -> lấy 6 từ b, res = [6, 7, 6]
          [] vs [0, 4]     -> lấy 0 từ b, res = [6, 7, 6, 0]
          [] vs [4]        -> lấy 4 từ b, res = [6, 7, 6, 0, 4]

merge a=[6, 7, 1], b=[6, 7, 9]
   [6, 7, 1] vs [6, 7, 9]  -> lấy 6 từ b, res = [6]
   [6, 7, 1] vs [7, 9]     -> lấy 7 từ b, res = [6, 7]
   [6, 7, 1] vs [9]        -> lấy 9 từ b, res = [6, 7, 9]
   [6, 7, 1] vs []         -> lấy 6 từ a, res = [6, 7, 9, 6]
      [7, 1] vs []         -> lấy 7 từ a, res = [6, 7, 9, 6, 7]
         [1] vs []         -> lấy 1 từ a, res = [6, 7, 9, 6, 7, 1]


[6, 7, 9, 6, 7, 1]

### Vì sao "so phần còn lại" đúng?

Trực giác ở trên đã cho thấy hai điều: (1) khi chữ số đầu khác nhau, quy tắc trùng với "lấy chữ số lớn hơn" — hiển nhiên đúng; (2) khi bằng nhau, nó đi về phía dãy sẽ **mở khóa chữ số lớn hơn sớm hơn**.

Chứng minh chặt chẽ cho trường hợp (2) dùng **lập luận trao đổi** (exchange argument): giả sử một cách trộn tối ưu lại lấy chữ số đầu từ dãy có phần còn lại nhỏ hơn, ta chỉ ra có thể "đổi vai" hai dãy trên đoạn tiền tố chung mà kết quả không giảm. Chứng minh này khá dài và nhiều trường hợp; ở mức nhập môn, ta hiểu trực giác và **kiểm chứng thực nghiệm** thật kỹ: so `merge_max` với brute force "thử mọi cách trộn" trên hàng nghìn cặp ngẫu nhiên, đồng thời đếm xem cách 2a sai bao nhiêu lần.

Quy tắc này cũng chính là lời giải chuẩn của bài **LeetCode 1754 — Largest Merge Of Two Strings** (bài tập 6 cuối notebook).

In [20]:
random.seed(1754)
wrong_head = 0
for trial in range(1500):
    a = [random.randint(0, 3) for _ in range(random.randint(0, 5))]
    b = [random.randint(0, 3) for _ in range(random.randint(0, 5))]
    ref = max(all_interleavings(a, b))
    assert merge_max(a, b) == ref, (a, b)
    if merge_head_only(a, b) != ref:
        wrong_head += 1
print("merge_max khớp brute force trên 1500 cặp ngẫu nhiên.")
print("merge_head_only sai", wrong_head, "/ 1500 lần.")

merge_max khớp brute force trên 1500 cặp ngẫu nhiên.
merge_head_only sai 229 / 1500 lần.


## 9. Cách 3 — Ghép lại: duyệt mọi `i` hợp lệ (cách chính)

Có đủ "linh kiện", ghép theo lộ trình ở mục 6:

```python
best = []
for i in range(max(0, k - n), min(k, m) + 1):
    cand = merge_max(max_subsequence(nums1, i), max_subsequence(nums2, k - i))
    if cand > best:
        best = cand
return best
```

- `best = []` lúc đầu: `[]` nhỏ hơn mọi list khác rỗng, nên ứng viên đầu tiên luôn thay được nó.
- Mọi `cand` đều dài đúng `k` (vì `i` nằm trong khoảng hợp lệ nên `max_subsequence` trả đủ độ dài) → so `cand > best` là so **giá trị số**, đúng ý đề.
- Vì sao phải thử **mọi** `i`, không đoán được? Example 1: `i = 2` thắng; nhưng ở các bộ dữ liệu khác, có thể chỉ lấy 0 chữ số từ `nums1`, hoặc lấy hết. Không có quy luật đơn giản → thử hết, mà số `i` thì ít.

In [21]:
def max_number(nums1: List[int], nums2: List[int], k: int) -> List[int]:
    m, n = len(nums1), len(nums2)
    best = []
    for i in range(max(0, k - n), min(k, m) + 1):
        cand = merge_max(max_subsequence(nums1, i), max_subsequence(nums2, k - i))
        if cand > best:
            best = cand
    return best

for nums1, nums2, k, expected in EXAMPLES:
    got = max_number(nums1, nums2, k)
    print(got, "OK" if got == expected else f"SAI, mong đợi {expected}")
    assert got == expected

[9, 8, 6, 5, 3] OK
[6, 7, 6, 0, 4] OK
[9, 8, 9] OK


### Trace đầy đủ Example 1

`nums1 = [3, 4, 6, 5]` (`m = 4`), `nums2 = [9, 1, 2, 5, 8, 3]` (`n = 6`), `k = 5` → `i` chạy `0..4`.

| `i` | `k − i` | `max_subsequence(nums1, i)` | `max_subsequence(nums2, k−i)` | `merge_max` | Cập nhật? | `best` |
|---:|---:|---|---|---|---|---|
| 0 | 5 | `[]` | `[9, 2, 5, 8, 3]` | `[9, 2, 5, 8, 3]` | **tốt nhất mới** | `[9, 2, 5, 8, 3]` |
| 1 | 4 | `[6]` | `[9, 5, 8, 3]` | `[9, 6, 5, 8, 3]` | **tốt nhất mới** | `[9, 6, 5, 8, 3]` |
| 2 | 3 | `[6, 5]` | `[9, 8, 3]` | `[9, 8, 6, 5, 3]` | **tốt nhất mới** | `[9, 8, 6, 5, 3]` |
| 3 | 2 | `[4, 6, 5]` | `[9, 8]` | `[9, 8, 4, 6, 5]` | — | `[9, 8, 6, 5, 3]` |
| 4 | 1 | `[3, 4, 6, 5]` | `[9]` | `[9, 3, 4, 6, 5]` | — | `[9, 8, 6, 5, 3]` |

Đọc bảng:

- `i = 0`: không lấy gì từ `nums1`, dãy con lớn nhất dài 5 của `nums2` là `[9, 2, 5, 8, 3]` (chỉ được xóa 1 chữ số → xóa `1`).
- `i = 1`: thêm `[6]` từ `nums1` → trộn ra `[9, 6, 5, 8, 3]`, lớn hơn vì `6 > 2` ở vị trí 2.
- `i = 2`: `[6, 5]` và `[9, 8, 3]` → `[9, 8, 6, 5, 3]` — **đáp án**.
- `i = 3, 4`: phải lấy nhiều chữ số nhỏ của `nums1` (`4`, `3`) → thua ở vị trí thứ 3.

In [22]:
def trace_max_number(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    best = []
    print(f"nums1={nums1}, nums2={nums2}, k={k}: i chạy {max(0, k - n)}..{min(k, m)}")
    for i in range(max(0, k - n), min(k, m) + 1):
        p1, p2 = max_subsequence(nums1, i), max_subsequence(nums2, k - i)
        cand = merge_max(p1, p2)
        better = cand > best
        if better:
            best = cand
        print(f"  i={i}: {str(p1):<13} + {str(p2):<16} -> {cand} {'<- tốt nhất mới' if better else ''}")
    print("  Đáp án:", best)
    return best

trace_max_number([3, 4, 6, 5], [9, 1, 2, 5, 8, 3], 5)
print()
trace_max_number([3, 9], [8, 9], 3)

nums1=[3, 4, 6, 5], nums2=[9, 1, 2, 5, 8, 3], k=5: i chạy 0..4
  i=0: []            + [9, 2, 5, 8, 3]  -> [9, 2, 5, 8, 3] <- tốt nhất mới
  i=1: [6]           + [9, 5, 8, 3]     -> [9, 6, 5, 8, 3] <- tốt nhất mới
  i=2: [6, 5]        + [9, 8, 3]        -> [9, 8, 6, 5, 3] <- tốt nhất mới
  i=3: [4, 6, 5]     + [9, 8]           -> [9, 8, 4, 6, 5] 
  i=4: [3, 4, 6, 5]  + [9]              -> [9, 3, 4, 6, 5] 
  Đáp án: [9, 8, 6, 5, 3]

nums1=[3, 9], nums2=[8, 9], k=3: i chạy 1..2
  i=1: [9]           + [8, 9]           -> [9, 8, 9] <- tốt nhất mới
  i=2: [3, 9]        + [9]              -> [9, 3, 9] 
  Đáp án: [9, 8, 9]


[9, 8, 9]

### Độ phức tạp cách 3

Đặt `m = len(nums1)`, `n = len(nums2)`. Số giá trị `i` là `min(k, m) − max(0, k − n) + 1`, nhiều nhất cỡ `min(m, n) + 1` (≤ 501).

Với mỗi `i`:

- `max_subsequence(nums1, i)` + `max_subsequence(nums2, k − i)`: `O(m + n)` (monotonic stack).
- `merge_max`: có `k` bước. Mỗi bước tạo hai lát cắt `a[i:]`, `b[j:]` (sao chép tới `O(k)` phần tử) và so sánh chúng (tới `O(k)` nếu trùng nhau rất dài) → `O(k)` mỗi bước → `O(k²)` cho cả lần trộn ở trường hợp xấu nhất.
- So `cand > best`: `O(k)`.

Tổng: **Time `O((min(m, n) + 1) · (m + n + k²))`** ở trường hợp xấu nhất. Bộ nhớ: **`O(m + n + k)`** (stack, kết quả, các lát cắt tạm).

Trên giấy `k²` nghe đáng sợ, nhưng thực tế: (1) phép cắt list và phép so sánh list chạy bằng C bên trong Python, rất nhanh; (2) khi dữ liệu không quá đặc biệt, so sánh dừng ngay ở vài chữ số đầu. Cell dưới đo thời gian ở kích thước lớn nhất của đề, cả dữ liệu ngẫu nhiên lẫn dữ liệu "toàn số 9" (mọi so sánh đều phải đi tới cuối).

In [23]:
import time

def timeit(fn, *args):
    t0 = time.perf_counter()
    fn(*args)
    return time.perf_counter() - t0

random.seed(500)
A = [random.randint(0, 9) for _ in range(500)]
B = [random.randint(0, 9) for _ in range(500)]
print(f"ngẫu nhiên, m=n=500, k=500 : {timeit(max_number, A, B, 500):.3f} s")
print(f"toàn số 9,  m=n=500, k=500 : {timeit(max_number, [9] * 500, [9] * 500, 500):.3f} s")
print(f"ngẫu nhiên, m=n=500, k=1000: {timeit(max_number, A, B, 1000):.3f} s (chỉ có i = 500)")

ngẫu nhiên, m=n=500, k=500 : 0.141 s


toàn số 9,  m=n=500, k=500 : 0.136 s
ngẫu nhiên, m=n=500, k=1000: 0.001 s (chỉ có i = 500)


## 10. Cách 4 (tùy chọn) — So sánh phần còn lại **không cần cắt list**

`a[i:] > b[j:]` mỗi bước **sao chép** hai lát cắt, kể cả khi chữ số đầu đã khác nhau (khi đó chỉ cần 1 phép so sánh!). Ta tự viết hàm so sánh bằng chỉ số:

```python
def greater(a, i, b, j):
    # True nếu a[i:] > b[j:] theo thứ tự từ điển
    while i < len(a) and j < len(b) and a[i] == b[j]:
        i += 1
        j += 1
    if j == len(b):        # b hết trước (hoặc cả hai cùng hết)
        return True
    if i == len(a):        # a hết trước, b còn -> a nhỏ hơn
        return False
    return a[i] > b[j]     # chỗ khác nhau đầu tiên quyết định
```

Giải thích các nhánh:

- Vòng `while` bỏ qua phần **trùng nhau**. Dừng khi một dãy hết hoặc gặp chữ số khác nhau.
- `j == len(b)`: `b[j:]` đã hết → `b[j:]` là tiền tố của `a[i:]` → `a` lớn hơn hoặc bằng. Trường hợp cả hai cùng hết (bằng nhau hoàn toàn) cũng trả `True`; lấy bên nào cũng được nên không sao.
- `i == len(a)` (và `b` còn): `a[i:]` là tiền tố thật sự của `b[j:]` → `a` nhỏ hơn.
- Còn lại: so chữ số khác nhau đầu tiên.

**Lợi ích:** không tạo list tạm (bộ nhớ phụ `O(1)` cho mỗi lần so sánh); khi chữ số đầu khác nhau, chỉ tốn 1 phép so sánh. **Big-O xấu nhất không đổi** (dữ liệu toàn số 9 vẫn phải so tới cuối), nhưng code chạy bằng Python thuần nên chưa chắc nhanh hơn bản cắt list chạy bằng C — ta đo để biết.

> Nâng cao (không cần cho bài này): có thể so `a[i:]` với `b[j:]` trong `O(1)` sau khi tiền xử lý **suffix array** của dãy `a + [-1] + b`, đưa phép trộn về `O(k)`. Với `k <= 1000` thì không đáng.

Bonus: một mẹo một dòng rất hay gặp trên LeetCode Discuss: `[max(a, b).pop(0) for _ in a + b]`. `max(a, b)` trả về **chính đối tượng list** lớn hơn (không phải bản sao), `.pop(0)` lấy chữ số đầu của nó ra — đúng quy tắc "so phần còn lại". Nó **thay đổi** `a`, `b` nên phải truyền bản sao, và `pop(0)` tốn `O(độ dài)`.

In [24]:
def greater(a, i, b, j):
    while i < len(a) and j < len(b) and a[i] == b[j]:
        i += 1
        j += 1
    if j == len(b):
        return True
    if i == len(a):
        return False
    return a[i] > b[j]

def merge_fast(a, b):
    i = j = 0
    res = []
    while i < len(a) or j < len(b):
        if greater(a, i, b, j):
            res.append(a[i]); i += 1
        else:
            res.append(b[j]); j += 1
    return res

def merge_oneliner(a, b):
    a, b = a[:], b[:]                     # sao chép vì pop(0) làm thay đổi list
    return [max(a, b).pop(0) for _ in a + b]

def max_number_fast(nums1, nums2, k):
    m, n = len(nums1), len(nums2)
    best = []
    for i in range(max(0, k - n), min(k, m) + 1):
        cand = merge_fast(max_subsequence(nums1, i), max_subsequence(nums2, k - i))
        if cand > best:
            best = cand
    return best

print(greater([6, 7], 0, [6, 0, 4], 0), greater([6], 0, [6, 7], 0), greater([5], 0, [5], 0))
for nums1, nums2, k, expected in EXAMPLES:
    assert max_number_fast(nums1, nums2, k) == expected
for a, b in [([6, 7], [6, 0, 4]), ([6, 7, 1], [6, 7, 9]), ([6], [6, 7])]:
    assert merge_fast(a, b) == merge_oneliner(a, b) == merge_max(a, b)
print("merge_fast, merge_oneliner, max_number_fast: OK")

True False True
merge_fast, merge_oneliner, max_number_fast: OK


In [25]:
for name, X, Y in [("ngẫu nhiên", A, B), ("toàn số 9", [9] * 500, [9] * 500)]:
    t_slice = timeit(max_number, X, Y, 500)
    t_fast = timeit(max_number_fast, X, Y, 500)
    print(f"{name:<11}: cắt list {t_slice:.3f} s | so bằng chỉ số {t_fast:.3f} s")

ngẫu nhiên : cắt list 0.147 s | so bằng chỉ số 0.121 s


toàn số 9  : cắt list 0.141 s | so bằng chỉ số 1.517 s


Đọc kết quả đo (con số cụ thể phụ thuộc máy):

- Với dữ liệu **ngẫu nhiên**, so sánh thường dừng sau 1–2 chữ số nên bản so bằng chỉ số không phải sao chép gì — nó thường nhanh hơn hoặc ngang bản cắt list.
- Với dữ liệu **toàn số 9**, mọi phép so sánh phải đi tới cuối. Lúc này bản cắt list (sao chép + so sánh bằng C) lại **nhanh hơn hẳn — thường gấp cả chục lần** — so với vòng `while` viết bằng Python thuần, vì mỗi phép so sánh phải đi qua hàng trăm chữ số bằng nhau.

Bài học: Big-O là như nhau; hằng số phụ thuộc vào việc phần lặp chạy trong C hay trong Python. Cả hai bản đều thừa sức qua giới hạn của đề, nên **bản nộp chính dùng `a[i:] > b[j:]`** cho gọn và dễ đọc. Bản `greater` đáng học vì nó dạy cách tự cài so sánh từ điển — kỹ năng cần ở các ngôn ngữ không có sẵn phép so sánh list.

## 11. Code nộp LeetCode

Chuyển cách 3 vào đúng khuôn `class Solution`. Hai bài con viết thành hàm lồng bên trong `maxNumber` cho gọn (LeetCode chỉ gọi `maxNumber`). Trên LeetCode, `List` đã được import sẵn; ở đây ta đã `from typing import List` ở đầu notebook.

In [26]:
class Solution:
    def maxNumber(self, nums1: List[int], nums2: List[int], k: int) -> List[int]:
        def pick(nums: List[int], t: int) -> List[int]:
            # dãy con lớn nhất độ dài t: monotonic stack với quyền xóa drop
            drop = len(nums) - t
            stack = []
            for x in nums:
                while drop > 0 and stack and stack[-1] < x:
                    stack.pop()
                    drop -= 1
                stack.append(x)
            return stack[:t]

        def merge(a: List[int], b: List[int]) -> List[int]:
            # trộn lớn nhất: so sánh PHẦN CÒN LẠI, không chỉ chữ số đầu
            i = j = 0
            res = []
            while i < len(a) or j < len(b):
                if a[i:] > b[j:]:
                    res.append(a[i]); i += 1
                else:
                    res.append(b[j]); j += 1
            return res

        m, n = len(nums1), len(nums2)
        best = []
        for i in range(max(0, k - n), min(k, m) + 1):
            cand = merge(pick(nums1, i), pick(nums2, k - i))
            if cand > best:
                best = cand
        return best


sol = Solution()
for nums1, nums2, k, expected in EXAMPLES:
    assert sol.maxNumber(nums1, nums2, k) == expected
print("3 ví dụ của đề: PASS")

3 ví dụ của đề: PASS


In [27]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS  {actual}")

check("k = m + n: lấy hết, chỉ còn trộn", sol.maxNumber([1, 2], [3], 3), [3, 1, 2])
check("k = 1: chữ số lớn nhất", sol.maxNumber([1, 5], [4, 2], 1), [5])
check("toàn số 0", sol.maxNumber([0, 0], [0], 2), [0, 0])
check("toàn chữ số bằng nhau", sol.maxNumber([9, 9], [9, 9, 9], 4), [9, 9, 9, 9])
check("Example 2 đổi chỗ (bẫy so chữ số đầu)", sol.maxNumber([6, 0, 4], [6, 7], 5), [6, 7, 6, 0, 4])
check("Example 3 đổi chỗ", sol.maxNumber([8, 9], [3, 9], 3), [9, 8, 9])
check("bằng nhau 2 chữ số rồi mới khác", sol.maxNumber([6, 7, 1], [6, 7, 9], 6), [6, 7, 9, 6, 7, 1])
check("mảng 1 phần tử", sol.maxNumber([5], [7], 1), [7])
check("trộn dài, nhiều lần hòa", sol.maxNumber([2, 5, 6, 4, 4, 0], [7, 3, 8, 0, 6, 5, 7, 6, 2], 15),
      [7, 3, 8, 2, 5, 6, 4, 4, 0, 6, 5, 7, 6, 2, 0])

k = m + n: lấy hết, chỉ còn trộn       PASS  [3, 1, 2]
k = 1: chữ số lớn nhất                 PASS  [5]
toàn số 0                              PASS  [0, 0]
toàn chữ số bằng nhau                  PASS  [9, 9, 9, 9]
Example 2 đổi chỗ (bẫy so chữ số đầu)  PASS  [6, 7, 6, 0, 4]
Example 3 đổi chỗ                      PASS  [9, 8, 9]
bằng nhau 2 chữ số rồi mới khác        PASS  [6, 7, 9, 6, 7, 1]
mảng 1 phần tử                         PASS  [7]
trộn dài, nhiều lần hòa                PASS  [7, 3, 8, 2, 5, 6, 4, 4, 0, 6, 5, 7, 6, 2, 0]


### Kiểm thử đối chiếu chéo với brute force

Test tay chỉ bao được những gì ta nghĩ tới. Để chắc chắn hơn, ta sinh **ngẫu nhiên** nhiều bộ `(nums1, nums2, k)` nhỏ (để brute force chạy nổi), cố định `random.seed` để lần nào chạy cũng ra cùng bộ test, rồi so **cả 4 phiên bản**: brute force (trọng tài), `max_number` (cách 3), `max_number_fast` (cách 4), `Solution().maxNumber`.

Một nửa số ca dùng chữ số `0..9`, một nửa dùng `0..2` — chữ số ít thì **hòa** nhiều, đúng chỗ dễ sai nhất của bước trộn.

In [28]:
random.seed(2026)
for trial in range(400):
    hi = 9 if trial % 2 == 0 else 2
    nums1 = [random.randint(0, hi) for _ in range(random.randint(1, 4))]
    nums2 = [random.randint(0, hi) for _ in range(random.randint(1, 4))]
    k = random.randint(1, len(nums1) + len(nums2))
    ref = max_number_bruteforce(nums1, nums2, k)
    assert len(ref) == k
    assert max_number(nums1, nums2, k) == ref, (nums1, nums2, k)
    assert max_number_fast(nums1, nums2, k) == ref, (nums1, nums2, k)
    assert sol.maxNumber(nums1, nums2, k) == ref, (nums1, nums2, k)
print("400 bộ ngẫu nhiên: brute force = cách 3 = cách 4 = Solution")

400 bộ ngẫu nhiên: brute force = cách 3 = cách 4 = Solution


## 12. Vì sao lời giải đúng — ghép các mắt xích

Lời giải đúng vì **bốn mắt xích**, mỗi mắt đã được lập luận hoặc kiểm chứng ở trên:

1. **Phân rã theo `i`.** Mọi đáp án hợp lệ gồm `i` chữ số từ `nums1` và `k − i` từ `nums2` với `max(0, k − n) <= i <= min(k, m)`. Thử mọi `i` trong khoảng này → không bỏ sót đáp án nào.
2. **Chỉ cần dãy con lớn nhất của mỗi bên.** Lập luận "cùng một kiểu trộn" (mục 6): thay một dãy con bằng dãy con lớn hơn cùng độ dài thì cách trộn tốt nhất không giảm. Nên với mỗi `i`, cặp `(max_subsequence(nums1, i), max_subsequence(nums2, k − i))` là tốt nhất.
3. **Bài con 1 đúng.** Monotonic stack thực hiện quy tắc tham lam "xóa chữ số đầu tiên nhỏ hơn chữ số sau nó, hết thì xóa ở cuối" `drop` lần; invariant: stack không tăng chừng nào còn quyền xóa.
4. **Bài con 2 đúng.** So **phần còn lại** `a[i:] > b[j:]`: chữ số đầu khác nhau → lấy cái lớn; bằng nhau → tiến theo dãy mở khóa chữ số lớn hơn sớm hơn (tiền tố ngắn hơn thì nhỏ hơn). Đã kiểm chứng với brute force trên hàng nghìn cặp.

Cuối cùng `best = max` của mọi ứng viên dài `k` → so từ điển = so giá trị số → đúng là số lớn nhất.

Điểm then chốt về **tư duy**: bài Hard này không đòi một ý tưởng "thần thánh" duy nhất; nó đòi **chia nhỏ** thành các bài con dễ hơn, giải từng bài con đúng, rồi chứng minh phép ghép không làm mất đáp án.

## 13. So sánh các cách

Ký hiệu: `m = len(nums1)`, `n = len(nums2)`, `I = min(k, m) − max(0, k − n) + 1 <= min(m, n) + 1` là số giá trị `i`.

| Cách | Ý tưởng | Time (xấu nhất) | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | mọi dãy con × mọi cách trộn | `Σ C(m,i)·C(n,k−i)·C(k,i)` × `O(k)` — hàm mũ | `O(C(k, i)·k)` cho danh sách cách trộn | chỉ cho input rất nhỏ; trọng tài |
| Bài con 1a. Cửa sổ | chọn max trong cửa sổ cho từng chữ số | `O(t·(m − t + 1))` ~ `O(m²)` | `O(t)` | dễ nghĩ, quét lặp |
| Bài con 1b. Monotonic stack | xóa chữ số nhỏ đứng trước chữ số lớn | `O(m)` amortized | `O(m)` | **dùng cái này** |
| Bài con 2a. So chữ số đầu | merge kiểu merge sort | `O(k)` | `O(k)` | **SAI** khi hòa |
| Bài con 2b. So phần còn lại | `a[i:] > b[j:]` | `O(k²)` | `O(k)` | đúng |
| 3. Ghép (cách chính) | thử mọi `i` + 1b + 2b | `O(I · (m + n + k²))` | `O(m + n + k)` | nộp bài, cỡ 0.1–0.3 s ở max constraints |
| 4. So bằng chỉ số | như 3 nhưng không cắt list | `O(I · (m + n + k²))` | `O(m + n + k)` | ít bộ nhớ tạm; nhanh khi dữ liệu ngẫu nhiên, nhưng chậm hơn nhiều lần khi toàn chữ số bằng nhau |

## 14. Lỗi người mới hay gặp

| # | Lỗi | Hậu quả | Sửa |
|---:|---|---|---|
| 1 | Trộn bằng cách chỉ so **chữ số đầu** `a[i] > b[j]` | sai khi hai chữ số đầu bằng nhau, ví dụ `[6, 0, 4]` + `[6, 7]` | so **phần còn lại** `a[i:] > b[j:]` |
| 2 | Vá lỗi 1 bằng cách so thêm **một** chữ số `a[i+1]` vs `b[j+1]` | vẫn sai khi bằng nhau 2 chữ số trở lên: `[6, 7, 1]` + `[6, 7, 9]` | so tới chỗ khác nhau đầu tiên (so cả phần còn lại) |
| 3 | Monotonic stack **quên cắt** `stack[:t]` | kết quả dài hơn `t` khi mảng giảm dần: `[8, 9, 7, 6]`, `t = 2` ra `[9, 7, 6]` | `return stack[:t]` |
| 4 | Monotonic stack **không giới hạn số lần pop** (bỏ `drop`) | kết quả ngắn hơn `t`: `[1, 2, 3, 4]`, `t = 2` ra `[4]` | đếm `drop = len(nums) − t`, chỉ pop khi `drop > 0` |
| 5 | Dùng `<=` thay `<` khi pop | phí quyền xóa vào chữ số bằng nhau: `[5, 5, 3]`, `t = 2` ra `[5, 3]` thay vì `[5, 5]` | pop khi `stack[-1] < x` |
| 6 | Duyệt `i` trong `range(k + 1)` thay vì khoảng hợp lệ | gọi bài con với `t > len(nums)`; tùy cách cài mà sai độ dài hoặc lỗi (cách cửa sổ: `max()` của cửa sổ rỗng → `ValueError`) | `range(max(0, k − n), min(k, m) + 1)` |
| 7 | So sánh ứng viên khác độ dài theo từ điển rồi tưởng là so số | `[9] > [1, 0, 0]` nhưng `9 < 100` | đảm bảo mọi ứng viên dài đúng `k` |

Cell dưới chạy các phiên bản sai để thấy tận mắt.

In [29]:
def merge_two_levels(a, b):                       # Lỗi 2: chỉ nhìn tới chữ số thứ hai
    i = j = 0
    res = []
    while i < len(a) or j < len(b):
        if j == len(b):
            take_a = True
        elif i == len(a):
            take_a = False
        elif a[i] != b[j]:
            take_a = a[i] > b[j]
        else:
            take_a = a[i + 1:i + 2] >= b[j + 1:j + 2]
        if take_a:
            res.append(a[i]); i += 1
        else:
            res.append(b[j]); j += 1
    return res

def pick_no_cut(nums, t):                          # Lỗi 3
    drop, st = len(nums) - t, []
    for x in nums:
        while drop > 0 and st and st[-1] < x:
            st.pop(); drop -= 1
        st.append(x)
    return st

def pick_no_drop(nums, t):                         # Lỗi 4
    st = []
    for x in nums:
        while st and st[-1] < x:
            st.pop()
        st.append(x)
    return st[:t]

def pick_le(nums, t):                              # Lỗi 5
    drop, st = len(nums) - t, []
    for x in nums:
        while drop > 0 and st and st[-1] <= x:
            st.pop(); drop -= 1
        st.append(x)
    return st[:t]

print("Lỗi 1:", merge_head_only([6, 0, 4], [6, 7]), "| đúng:", merge_max([6, 0, 4], [6, 7]))
print("Lỗi 2:", merge_two_levels([6, 7, 1], [6, 7, 9]), "| đúng:", merge_max([6, 7, 1], [6, 7, 9]))
print("Lỗi 3:", pick_no_cut([8, 9, 7, 6], 2), "| đúng:", max_subsequence([8, 9, 7, 6], 2))
print("Lỗi 4:", pick_no_drop([1, 2, 3, 4], 2), "| đúng:", max_subsequence([1, 2, 3, 4], 2))
print("Lỗi 5:", pick_le([5, 5, 3], 2), "| đúng:", max_subsequence([5, 5, 3], 2))
try:
    max_subseq_window([3, 9], 3)                   # Lỗi 6: t > len(nums)
except ValueError as e:
    print("Lỗi 6: ValueError ->", e)
print("Lỗi 7:", [9] > [1, 0, 0], "nhưng", 9 > 100)

Lỗi 1: [6, 6, 7, 0, 4] | đúng: [6, 7, 6, 0, 4]
Lỗi 2: [6, 7, 6, 7, 9, 1] | đúng: [6, 7, 9, 6, 7, 1]
Lỗi 3: [9, 7, 6] | đúng: [9, 7]
Lỗi 4: [4] | đúng: [3, 4]
Lỗi 5: [5, 3] | đúng: [5, 5]
Lỗi 6: ValueError -> max() iterable argument is empty
Lỗi 7: True nhưng False


## 15. Mẹo nhận dạng pattern và các bài liên quan

**Dấu hiệu 1 — "chọn `t` phần tử, giữ thứ tự, lớn nhất / nhỏ nhất theo từ điển".** Nghĩ ngay tới **monotonic stack với ngân sách xóa** `drop = len − t`:

- muốn **lớn nhất** → pop khi `stack[-1] < x`;
- muốn **nhỏ nhất** → pop khi `stack[-1] > x`;
- luôn nhớ: chỉ pop khi `drop > 0`, cuối cùng cắt `stack[:t]`.

**Dấu hiệu 2 — "trộn hai dãy, giữ thứ tự mỗi dãy, kết quả lớn nhất theo từ điển".** So **phần còn lại** `a[i:] > b[j:]`, không chỉ phần tử đầu.

**Dấu hiệu 3 — "hai nguồn, tổng số lượng cố định `k`".** Liệt kê cách chia `i` + `(k − i)`, giải từng phía độc lập, rồi ghép. Nhớ tính đúng khoảng `max(0, k − n) .. min(k, m)`.

**Bài liên quan:**

| Bài | Liên hệ |
|---|---|
| 402. Remove K Digits (Medium) | bài con 1 nhưng **nhỏ nhất**, input là chuỗi, phải bỏ số 0 ở đầu |
| 1673. Find the Most Competitive Subsequence (Medium) | dãy con **nhỏ nhất** độ dài `k` — bài tập 5 |
| 316. Remove Duplicate Letters / 1081. Smallest Subsequence of Distinct Characters (Medium) | monotonic stack + điều kiện "còn xuất hiện phía sau mới được pop" |
| 2030. Smallest K-Length Subsequence With Occurrences of a Letter (Hard) | monotonic stack với **nhiều ngân sách** cùng lúc |
| 1754. Largest Merge Of Two Strings (Medium) | **đúng** bài con 2 trên chuỗi — bài tập 6 |
| 179. Largest Number (Medium) | ghép số lớn nhất, so sánh theo `a + b` vs `b + a` |

Cell dưới giải luôn LeetCode 402 bằng đúng "bộ khung" của bài con 1, chỉ đảo dấu so sánh.

In [30]:
def remove_k_digits(num: str, k: int) -> str:
    # LeetCode 402: xóa k chữ số để được số NHỎ NHẤT
    stack = []
    for ch in num:
        while k > 0 and stack and stack[-1] > ch:      # đảo dấu so với bài 321
            stack.pop()
            k -= 1
        stack.append(ch)
    if k > 0:
        stack = stack[:-k]                             # còn quyền xóa -> xóa ở đuôi
    return "".join(stack).lstrip("0") or "0"

print(remove_k_digits("1432219", 3))   # "1219"
print(remove_k_digits("10200", 1))     # "200"
print(remove_k_digits("10", 2))        # "0"

1219
200
0


## 16. Bài tập vận dụng

Làm theo thứ tự, **đừng xem đáp án** ở cuối trước khi tự thử. Gợi ý theo 3 mức ở mục 17.

### Bài 1 — Đoán output (so sánh list)

Không chạy code, đoán kết quả từng dòng:

```python
print([9, 8] > [9, 7, 9])
print([6, 7] > [6, 7, 0])
print([] < [0])
print([3, 3] == [3, 3])
print(max([[5, 1], [4, 9, 9], [5, 0, 9]]))
print([1, 0, 0] > [9])
print([6, 0, 4] > [6, 7])
```

### Bài 2 — Tính bằng tay

**(a)** Lập bảng trace monotonic stack (cột `i`, `x`, pop, stack sau, `drop` còn) cho `max_subsequence([4, 9, 0, 2, 7], 3)`. Kết quả là gì?

**(b)** Trộn `a = [3, 1]` và `b = [3, 5]` bằng quy tắc "so phần còn lại", ghi từng bước. Sau đó trộn lại bằng quy tắc sai "chỉ so chữ số đầu, bằng nhau lấy `a`". Hai kết quả khác nhau thế nào?

### Bài 3 — Điền TODO: dãy con lớn nhất độ dài `t`

Hoàn thiện hàm theo các bước TODO. Khi xong, bỏ comment các dòng `assert` để tự kiểm tra.

In [31]:
def exercise_max_subsequence(nums: List[int], t: int) -> List[int]:
    # TODO 1: tính drop = số chữ số được phép xóa
    # TODO 2: tạo stack rỗng
    # TODO 3: duyệt từng x trong nums:
    #         trong khi còn quyền xóa, stack khác rỗng và đỉnh stack < x: pop, giảm drop
    # TODO 4: đẩy x vào stack
    # TODO 5: trả về t phần tử đầu của stack
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_max_subsequence([9, 1, 2, 5, 8, 3], 3) == [9, 8, 3]
# assert exercise_max_subsequence([8, 9, 7, 6], 2) == [9, 7]
# assert exercise_max_subsequence([5, 5, 3], 2) == [5, 5]
# assert exercise_max_subsequence([1, 2], 0) == []

### Bài 4 — Tự viết: trộn lớn nhất **không dùng lát cắt**

Viết `exercise_greater(a, i, b, j)` trả `True` khi `a[i:] > b[j:]` theo thứ tự từ điển **mà không tạo lát cắt** (không dùng `a[i:]`), rồi dùng nó viết `exercise_merge(a, b)`. Cẩn thận ba trường hợp: `a` hết trước, `b` hết trước, gặp chữ số khác nhau.

In [32]:
def exercise_greater(a: List[int], i: int, b: List[int], j: int) -> bool:
    # TODO: bỏ qua phần trùng nhau, rồi xử lý 3 trường hợp
    raise NotImplementedError

def exercise_merge(a: List[int], b: List[int]) -> List[int]:
    # TODO: hai con trỏ i, j; mỗi bước dùng exercise_greater để chọn
    raise NotImplementedError

# assert exercise_greater([6, 7], 0, [6, 0, 4], 0) is True
# assert exercise_greater([6], 0, [6, 7], 0) is False
# assert exercise_merge([6, 0, 4], [6, 7]) == [6, 7, 6, 0, 4]
# assert exercise_merge([6, 7, 1], [6, 7, 9]) == [6, 7, 9, 6, 7, 1]
# assert exercise_merge([], [1, 2]) == [1, 2]

### Bài 5 — Biến thể: LeetCode 1673 — Find the Most Competitive Subsequence (Medium)

Cho mảng số nguyên `nums` và số nguyên dương `k`, trả về dãy con **"cạnh tranh nhất"** (most competitive) độ dài `k` của `nums`. Dãy `a` cạnh tranh hơn dãy `b` (cùng độ dài) nếu tại vị trí đầu tiên khác nhau, `a` có số **nhỏ hơn** — tức là dãy con **nhỏ nhất theo thứ tự từ điển**.

- `nums = [3, 5, 2, 6]`, `k = 2` → `[2, 6]`
- `nums = [2, 4, 3, 3, 5, 4, 9, 6]`, `k = 4` → `[2, 3, 3, 4]`

Constraints: `1 <= nums.length <= 10^5`, `0 <= nums[i] <= 10^9`, `1 <= k <= nums.length`. (Lưu ý: giá trị không còn là chữ số 0..9, nhưng thuật toán không quan tâm.)

In [33]:
def exercise_most_competitive(nums: List[int], k: int) -> List[int]:
    # TODO: monotonic stack như bài con 1, nhưng muốn NHỎ NHẤT
    pass

# assert exercise_most_competitive([3, 5, 2, 6], 2) == [2, 6]
# assert exercise_most_competitive([2, 4, 3, 3, 5, 4, 9, 6], 4) == [2, 3, 3, 4]

### Bài 6 — Mini challenge: LeetCode 1754 — Largest Merge Of Two Strings (Medium)

Cho hai chuỗi `word1`, `word2`. Bạn xây chuỗi `merge` bằng cách lặp lại: chọn một trong hai chuỗi còn khác rỗng, cắt **ký tự đầu** của nó và nối vào cuối `merge`, cho tới khi cả hai rỗng. Trả về chuỗi `merge` **lớn nhất theo thứ tự từ điển**.

- `word1 = "cabaa"`, `word2 = "bcaaa"` → `"cbcabaaaaa"`
- `word1 = "abcabc"`, `word2 = "abdcaba"` → `"abdcabcabcaba"`

Constraints: `1 <= word1.length, word2.length <= 3000`, chỉ gồm chữ cái thường.

Câu hỏi phụ: so sánh chuỗi trong Python có theo cùng quy tắc từ điển như so sánh list không?

**Bonus**: đóng notebook lại và tự viết lại toàn bộ `maxNumber` của bài 321 từ đầu (ba hàm: `pick`, `merge`, vòng lặp `i`), rồi chạy lại với 3 ví dụ của đề.

In [34]:
def exercise_largest_merge(word1: str, word2: str) -> str:
    # TODO: hai con trỏ; mỗi bước so phần còn lại của hai chuỗi
    pass

# assert exercise_largest_merge("cabaa", "bcaaa") == "cbcabaaaaa"
# assert exercise_largest_merge("abcabc", "abdcaba") == "abdcabcabcaba"

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: so từng vị trí từ trái sang; vị trí khác nhau đầu tiên quyết định.
- Mức 2: nếu một list là tiền tố của list kia thì list ngắn hơn nhỏ hơn. `[]` là tiền tố của mọi list.
- Mức 3: dòng `max(...)`: cả `[5, 1]` và `[5, 0, 9]` đều bắt đầu bằng 5; so tiếp `1` với `0`.

**Bài 2**
- Mức 1: (a) `drop = 5 − 3 = 2`. (b) ở bước đầu, cả hai dãy đều bắt đầu bằng `3`.
- Mức 2: (a) mỗi `x` mới: nhìn đỉnh stack, nhỏ hơn `x` và còn quyền xóa thì pop. (b) so `[3, 1]` với `[3, 5]`: khác ở vị trí 1.
- Mức 3: (a) `9` sẽ đẩy `4` ra; sau đó `2` đẩy `0` ra và hết quyền xóa. (b) quy tắc đúng lấy `3` của `b` trước để `5` lộ ra.

**Bài 3**
- Mức 1: chép lại đúng khung ở mục 7, bước 1b.
- Mức 2: điều kiện `while` có **ba** phần nối bằng `and`; thiếu phần nào cũng sai (xem mục Lỗi hay gặp 3, 4, 5).
- Mức 3: `while drop > 0 and stack and stack[-1] < x: stack.pop(); drop -= 1`.

**Bài 4**
- Mức 1: phần trùng nhau không quyết định gì — bỏ qua bằng `while`.
- Mức 2: sau `while`, kiểm tra `j == len(b)` **trước** `i == len(a)` để trường hợp hai bên bằng nhau hoàn toàn trả `True`.
- Mức 3: mục 10 có hàm `greater`; hãy tự viết trước rồi mới so.

**Bài 5**
- Mức 1: "nhỏ nhất" → đảo dấu so sánh khi pop.
- Mức 2: `drop = len(nums) − k`.
- Mức 3: `while drop > 0 and stack and stack[-1] > x: ...`, cuối cùng `stack[:k]`.

**Bài 6**
- Mức 1: đây chính là bài con 2, chỉ là chuỗi thay vì list.
- Mức 2: Python so chuỗi theo từ điển giống hệt list (so từng ký tự, tiền tố ngắn hơn thì nhỏ hơn).
- Mức 3: `if word1[i:] > word2[j:]: lấy word1[i]`; nối các ký tự vào list rồi `"".join(...)`.

## 18. Đáp án tham khảo

### Bài 1

```text
True             # 8 > 7 ở vị trí 1
False            # [6, 7] là tiền tố của [6, 7, 0] -> nhỏ hơn
True             # [] nhỏ hơn mọi list khác rỗng
True
[5, 1]           # [5, 1] > [5, 0, 9] (1 > 0) > [4, 9, 9]
False            # 1 < 9 ở vị trí 0 (dù 100 > 9 về giá trị)
False            # 0 < 7 ở vị trí 1
```

### Bài 2

**(a)** `max_subsequence([4, 9, 0, 2, 7], 3)`, `drop = 2`:

| i | x | pop | stack sau | drop còn |
|---:|---:|---|---|---:|
| 0 | 4 | — | `[4]` | 2 |
| 1 | 9 | pop `[4]` | `[9]` | 1 |
| 2 | 0 | — | `[9, 0]` | 1 |
| 3 | 2 | pop `[0]` | `[9, 2]` | 0 |
| 4 | 7 | — | `[9, 2, 7]` | 0 |

Kết quả `[9, 2, 7]`. Kiểm tra lại bằng lý: chữ số đầu phải là `9` (lớn nhất có thể), sau `9` còn `[0, 2, 7]` phải chọn 2 → `[2, 7]`.

**(b)** Quy tắc đúng (so phần còn lại):

| Bước | `a[i:]` | `b[j:]` | So sánh | Lấy | Kết quả |
|---:|---|---|---|---|---|
| 1 | `[3, 1]` | `[3, 5]` | `a[i:] <= b[j:]` | b → **3** | `[3]` |
| 2 | `[3, 1]` | `[5]` | `a[i:] <= b[j:]` | b → **5** | `[3, 5]` |
| 3 | `[3, 1]` | `[]` | `a[i:] > b[j:]` | a → **3** | `[3, 5, 3]` |
| 4 | `[1]` | `[]` | `a[i:] > b[j:]` | a → **1** | `[3, 5, 3, 1]` |

Quy tắc sai (chỉ so chữ số đầu, hòa lấy `a`): lấy `3` của `a` → `[1]` vs `[3, 5]` → `3`, `5`, rồi `1` → `[3, 3, 5, 1]`, nhỏ hơn `[3, 5, 3, 1]` ở vị trí 1.

In [35]:
# Kiểm chứng Bài 1 và Bài 2
print([9, 8] > [9, 7, 9], [6, 7] > [6, 7, 0], [] < [0], [3, 3] == [3, 3])
print(max([[5, 1], [4, 9, 9], [5, 0, 9]]), [1, 0, 0] > [9], [6, 0, 4] > [6, 7])

assert max_subsequence([4, 9, 0, 2, 7], 3) == [9, 2, 7] == max_subseq_bruteforce([4, 9, 0, 2, 7], 3)
assert merge_max([3, 1], [3, 5]) == [3, 5, 3, 1] == max(all_interleavings([3, 1], [3, 5]))
assert merge_head_only([3, 1], [3, 5]) == [3, 3, 5, 1]
print("Bài 2: OK")

True False True True
[5, 1] False False
Bài 2: OK


In [36]:
# Đáp án Bài 3
def exercise_max_subsequence(nums: List[int], t: int) -> List[int]:
    drop = len(nums) - t
    stack = []
    for x in nums:
        while drop > 0 and stack and stack[-1] < x:
            stack.pop()
            drop -= 1
        stack.append(x)
    return stack[:t]

assert exercise_max_subsequence([9, 1, 2, 5, 8, 3], 3) == [9, 8, 3]
assert exercise_max_subsequence([8, 9, 7, 6], 2) == [9, 7]
assert exercise_max_subsequence([5, 5, 3], 2) == [5, 5]
assert exercise_max_subsequence([1, 2], 0) == []

# Đáp án Bài 4
def exercise_greater(a: List[int], i: int, b: List[int], j: int) -> bool:
    while i < len(a) and j < len(b) and a[i] == b[j]:
        i += 1
        j += 1
    if j == len(b):          # b hết trước hoặc bằng nhau hoàn toàn
        return True
    if i == len(a):          # a là tiền tố thật sự của b
        return False
    return a[i] > b[j]

def exercise_merge(a: List[int], b: List[int]) -> List[int]:
    i = j = 0
    res = []
    while i < len(a) or j < len(b):
        if exercise_greater(a, i, b, j):
            res.append(a[i]); i += 1
        else:
            res.append(b[j]); j += 1
    return res

assert exercise_greater([6, 7], 0, [6, 0, 4], 0) is True
assert exercise_greater([6], 0, [6, 7], 0) is False
assert exercise_merge([6, 0, 4], [6, 7]) == [6, 7, 6, 0, 4]
assert exercise_merge([6, 7, 1], [6, 7, 9]) == [6, 7, 9, 6, 7, 1]
assert exercise_merge([], [1, 2]) == [1, 2]

random.seed(4)
for _ in range(500):
    a = [random.randint(0, 2) for _ in range(random.randint(0, 6))]
    b = [random.randint(0, 2) for _ in range(random.randint(0, 6))]
    assert exercise_merge(a, b) == merge_max(a, b)
print("Bài 3, Bài 4: PASS (kèm 500 cặp ngẫu nhiên cho Bài 4)")

Bài 3, Bài 4: PASS (kèm 500 cặp ngẫu nhiên cho Bài 4)


In [37]:
# Đáp án Bài 5 — LeetCode 1673
def exercise_most_competitive(nums: List[int], k: int) -> List[int]:
    drop = len(nums) - k
    stack = []
    for x in nums:
        while drop > 0 and stack and stack[-1] > x:     # muốn NHỎ NHẤT -> pop khi đỉnh lớn hơn
            stack.pop()
            drop -= 1
        stack.append(x)
    return stack[:k]

assert exercise_most_competitive([3, 5, 2, 6], 2) == [2, 6]
assert exercise_most_competitive([2, 4, 3, 3, 5, 4, 9, 6], 4) == [2, 3, 3, 4]

random.seed(1673)
for _ in range(300):
    nums = [random.randint(0, 4) for _ in range(random.randint(1, 7))]
    k = random.randint(1, len(nums))
    ref = min([nums[p] for p in idx] for idx in combinations(range(len(nums)), k))
    assert exercise_most_competitive(nums, k) == ref

# Đáp án Bài 6 — LeetCode 1754 (chuỗi so sánh từ điển giống hệt list)
def exercise_largest_merge(word1: str, word2: str) -> str:
    i = j = 0
    out = []
    while i < len(word1) or j < len(word2):
        if word1[i:] > word2[j:]:
            out.append(word1[i]); i += 1
        else:
            out.append(word2[j]); j += 1
    return "".join(out)

assert exercise_largest_merge("cabaa", "bcaaa") == "cbcabaaaaa"
assert exercise_largest_merge("abcabc", "abdcaba") == "abdcabcabcaba"
print("Bài 5 (kèm 300 ca so với brute force), Bài 6: PASS")

Bài 5 (kèm 300 ca so với brute force), Bài 6: PASS


## 19. Template ngắn nên nhớ

```python
def max_subsequence(nums, t):              # dãy con LỚN NHẤT độ dài t
    drop, stack = len(nums) - t, []
    for x in nums:
        while drop > 0 and stack and stack[-1] < x:   # nhỏ nhất: đổi thành >
            stack.pop(); drop -= 1
        stack.append(x)
    return stack[:t]                       # đừng quên cắt

def merge_max(a, b):                       # trộn LỚN NHẤT, giữ thứ tự mỗi dãy
    i = j = 0; res = []
    while i < len(a) or j < len(b):
        if a[i:] > b[j:]: res.append(a[i]); i += 1    # so PHẦN CÒN LẠI
        else:             res.append(b[j]); j += 1
    return res

best = []
for i in range(max(0, k - n), min(k, m) + 1):          # mọi cách chia hợp lệ
    best = max(best, merge_max(max_subsequence(nums1, i), max_subsequence(nums2, k - i)))
```

Ba câu thần chú: **chia `i` + `(k − i)`** → **stack có ngân sách xóa** → **trộn bằng so sánh phần còn lại**.

## Checklist trước khi submit

- [ ] Khoảng `i` là `max(0, k − n) .. min(k, m)` (có `+ 1` ở cận trên của `range`).
- [ ] Monotonic stack: điều kiện `drop > 0 and stack and stack[-1] < x` (dấu `<` chứ không phải `<=`).
- [ ] Trả `stack[:t]`, không trả cả `stack`.
- [ ] Trộn bằng `a[i:] > b[j:]` (hoặc hàm `greater` tự viết), **không** chỉ so `a[i]` với `b[j]`.
- [ ] Vòng trộn chạy `while i < len(a) or j < len(b)` — dùng `or` để lấy hết cả hai dãy.
- [ ] `best` khởi tạo `[]`; so sánh ứng viên cùng độ dài `k`.
- [ ] Đã thử Example 2 **đổi chỗ** hai mảng (`[6,0,4]`, `[6,7]`) — bẫy kinh điển của bước trộn.
- [ ] Đã thử `k = m + n`, `k = 1`, toàn chữ số bằng nhau.
- [ ] Đúng signature `maxNumber(self, nums1: List[int], nums2: List[int], k: int) -> List[int]`, trả về **list**, không trả `int`.